# Emerging Technologies — Problems

## Problem 1: Generating Random Boolean Functions

This problem asks for a Python function `random_constant_balanced` that returns a
randomly chosen function from the set of **constant** or **balanced** Boolean
functions taking four Boolean arguments as input.

### Background

The Deutsch–Jozsa algorithm [1] is one of the earliest examples of a quantum
algorithm that provides a provable speedup over deterministic classical
algorithms. It is designed to decide, with a single query, whether a given
Boolean function $f : \{0,1\}^n \to \{0,1\}$ belongs to one of two restricted
classes:

- **Constant functions** — $f(x) = c$ for every input $x$, where $c \in \{0,1\}$.
- **Balanced functions** — $f$ returns $0$ on exactly half of the $2^n$ possible
  inputs and $1$ on the other half.

A general Boolean function need not be either of these; in fact, most are
neither. The Deutsch–Jozsa problem is restricted *by promise* to only these two
classes. The modern textbook formulation of the algorithm given by Cleve,
Ekert, Macchiavello and Mosca [2] is the one used in most quantum-computing
tutorials today, including the IBM Quantum Learning material [3] linked in the
problem statement.

Before we can simulate or analyse the algorithm in later problems, we need a
way to generate such functions at random — that is the goal of Problem 1.

### References

[1] D. Deutsch and R. Jozsa, "Rapid solution of problems by quantum
computation," *Proceedings of the Royal Society A*, vol. 439, no. 1907,
pp. 553–558, 1992. https://doi.org/10.1098/rspa.1992.0167

[2] R. Cleve, A. Ekert, C. Macchiavello, and M. Mosca, "Quantum algorithms
revisited," *Proceedings of the Royal Society A*, vol. 454, no. 1969,
pp. 339–354, 1998. https://doi.org/10.1098/rspa.1998.0164

[3] IBM Quantum Learning, "The Deutsch–Jozsa algorithm."
https://quantum.cloud.ibm.com/learning/en/modules/computer-science/deutsch-jozsa

### Counting constant and balanced functions on four inputs

A Boolean function $f : \{0,1\}^4 \to \{0,1\}$ is fully specified by its
**truth table** — the list of outputs for each of the $2^4 = 16$ possible
input combinations. Since each of those 16 outputs is independently $0$ or
$1$, there are $2^{16} = 65{,}536$ Boolean functions on four inputs in total
[1].

We are interested in two specific subsets:

**Constant functions.** There are exactly **two**: the function that returns
$0$ everywhere, and the function that returns $1$ everywhere.

**Balanced functions.** A balanced function returns $1$ on exactly half of
the $2^4 = 16$ inputs, and $0$ on the other half. The number of such
functions is therefore the number of ways to choose which 8 of the 16 input
combinations map to $1$:

$$
\binom{16}{8} = 12{,}870.
$$

So the *promise set* for the four-input Deutsch–Jozsa problem contains
$2 + 12{,}870 = 12{,}872$ functions in total. Out of all $65{,}536$ Boolean
functions on four inputs, only about **19.6%** satisfy the Deutsch–Jozsa
promise — the rest are neither constant nor balanced and the algorithm is
not defined for them [2].

This count drives the implementation strategy: rather than rejection-sampling
from all $2^{16}$ functions (which would discard roughly four out of every
five candidates), we will sample directly from the constant and balanced
classes in proportion to their sizes.

### References

[1] D. E. Knuth, *The Art of Computer Programming, Volume 4A: Combinatorial
Algorithms, Part 1*. Upper Saddle River, NJ: Addison-Wesley, 2011, ch. 7.1.1.

[2] M. A. Nielsen and I. L. Chuang, *Quantum Computation and Quantum
Information*, 10th anniversary ed. Cambridge: Cambridge University Press,
2010, sec. 1.4.3.

### Representation strategy: truth tables as integers

A Boolean function on four inputs is fully described by its 16-entry truth
table. There are several reasonable ways to represent such a table in
Python, but the most compact is a single 16-bit integer where bit $i$
holds the output $f(x)$ for the input $x$ whose binary encoding is $i$.

For example, the integer `0b1010101010101010` (which is `0xAAAA`, or
$43{,}690$ in decimal) encodes the function

$$
f(x_3, x_2, x_1, x_0) = x_0,
$$

because its truth table outputs $1$ exactly when the least significant
input bit is $1$. This is a balanced function: half of the 16 inputs have
$x_0 = 1$.

This representation has three advantages relevant to our task [1]:

1. **Constant functions are trivial to construct.** The all-zeros function
   is the integer $0$ and the all-ones function is $2^{16} - 1 = 65{,}535$.
2. **Balancedness is a single popcount.** A function is balanced on four
   inputs if and only if its 16-bit truth table has exactly 8 bits set,
   which Python exposes directly as `int.bit_count()` (added in Python 3.10).
3. **Sampling a balanced function reduces to a random bit selection.** We
   can pick which 8 of the 16 input positions map to $1$ uniformly at
   random using `random.sample`, then assemble the integer.

The actual *callable* the user receives will wrap this integer in a closure
that takes four Boolean arguments, packs them into the index $i$, and
returns the corresponding bit of the truth table.

### References

[1] H. S. Warren Jr., *Hacker's Delight*, 2nd ed. Upper Saddle River, NJ:
Addison-Wesley, 2013, ch. 5 ("Counting Bits").

In [114]:
"""Imports and helpers for Problem 1."""

import random
from typing import Callable

# A Boolean function on four inputs maps four bools to a single bool.
BoolFunc4 = Callable[[bool, bool, bool, bool], bool]


def _truth_table_to_callable(table: int) -> BoolFunc4:
    """Wrap a 16-bit truth table integer in a 4-argument Boolean callable.

    Bit ``i`` of ``table`` is the output of the function on the input whose
    binary encoding (with ``x3`` as the most significant bit) equals ``i``.

    Parameters
    ----------
    table : int
        A non-negative integer in the range ``[0, 2**16)`` whose binary
        representation is the truth table of the function.

    Returns
    -------
    Callable[[bool, bool, bool, bool], bool]
        A function ``f(x3, x2, x1, x0)`` that returns the corresponding
        truth-table bit as a Python ``bool``.
    """
    if not 0 <= table < (1 << 16):
        raise ValueError(
            f"truth table must fit in 16 bits, got {table}"
        )

    def f(x3: bool, x2: bool, x1: bool, x0: bool) -> bool:
        # Pack the four input bits into an index in [0, 16).
        index = (int(bool(x3)) << 3) | (int(bool(x2)) << 2) \
                | (int(bool(x1)) << 1) | int(bool(x0))
        return bool((table >> index) & 1)

    return f

### Implementing `random_constant_balanced`

We now have everything we need: a representation (16-bit truth table), a
wrapper that turns such a table into a callable, and the counts of each
class (2 constant functions, $\binom{16}{8} = 12{,}870$ balanced functions).

The function below samples uniformly from the union of these two sets. To
keep the sampling truly uniform across all $12{,}872$ functions we weight
the choice between the two branches by their class sizes — otherwise a
50/50 coin flip between "constant" and "balanced" would massively
over-represent the two constant functions.

Within each branch, the sampling is straightforward:

- **Constant branch.** Choose the all-zeros or all-ones truth table with
  equal probability.
- **Balanced branch.** Choose 8 of the 16 input positions uniformly at
  random using `random.sample`, then set those bits in the truth table.

In [116]:
def random_constant_balanced(rng: random.Random | None = None) -> BoolFunc4:
    """Return a uniformly random constant or balanced Boolean function on 4 inputs.

    The returned callable accepts four Boolean arguments and returns a single
    Boolean output. The function is drawn uniformly at random from the union
    of:

    - the 2 constant functions on 4 inputs (always ``False``, always ``True``), and
    - the C(16, 8) = 12,870 balanced functions on 4 inputs.

    The branch (constant vs. balanced) is selected with probability proportional
    to the size of each class, so every one of the 12,872 functions in the
    promise set is equally likely to be returned.

    Parameters
    ----------
    rng : random.Random, optional
        A random number generator. If ``None``, the module-level ``random``
        functions are used. Passing an explicit ``random.Random`` instance
        allows the caller to seed the sampling for reproducibility.

    Returns
    -------
    Callable[[bool, bool, bool, bool], bool]
        A Boolean function that is either constant or balanced.
    """
    # Use the supplied generator, or fall back to the module-level one.
    choice = rng.choice if rng is not None else random.choice
    sample = rng.sample if rng is not None else random.sample
    random_func = rng.random if rng is not None else random.random

    # There are 2 constant functions and C(16, 8) = 12,870 balanced functions.
    # Probability of the constant branch = 2 / 12,872.
    n_constant = 2
    n_balanced = 12_870
    p_constant = n_constant / (n_constant + n_balanced)

    if random_func() < p_constant:
        # Constant branch: 0x0000 (always False) or 0xFFFF (always True).
        table = choice([0x0000, 0xFFFF])
    else:
        # Balanced branch: pick 8 of the 16 input positions to map to True.
        ones_positions = sample(range(16), 8)
        table = 0
        for pos in ones_positions:
            table |= (1 << pos)

    return _truth_table_to_callable(table)

### Verifying the implementation

Before using `random_constant_balanced` anywhere else, we should confirm that
it actually upholds the promise. Two checks are appropriate:

1. **Per-sample check.** Every function returned must be either constant
   (all 16 outputs equal) or balanced (exactly 8 outputs are `True`). We
   evaluate the function on all 16 inputs and verify this directly.

2. **Distributional check.** Over many samples, the empirical proportion of
   constant functions should approach the theoretical value
   $2 / 12{,}872 \approx 0.0155\%$. Because constants are so rare, a small
   sample size will not detect bias reliably; we use a large sample and
   allow a generous tolerance.

These are written as plain `assert` statements rather than as a `pytest`
suite so the notebook stays self-contained and reproducible by anyone who
clones the repository, but the structure mirrors what a `pytest` test
module would look like.

In [118]:
def _evaluate_truth_table(f: BoolFunc4) -> list[bool]:
    """Evaluate a 4-input Boolean function on all 16 inputs.

    Returns the outputs in index order, where index ``i`` corresponds to
    the input ``(x3, x2, x1, x0)`` whose binary encoding equals ``i``.
    """
    outputs = []
    for i in range(16):
        x3 = bool((i >> 3) & 1)
        x2 = bool((i >> 2) & 1)
        x1 = bool((i >> 1) & 1)
        x0 = bool(i & 1)
        outputs.append(f(x3, x2, x1, x0))
    return outputs


def _classify(f: BoolFunc4) -> str:
    """Return 'constant', 'balanced', or 'neither' for a 4-input function."""
    outputs = _evaluate_truth_table(f)
    n_true = sum(outputs)
    if n_true == 0 or n_true == 16:
        return "constant"
    if n_true == 8:
        return "balanced"
    return "neither"


# --- Per-sample check ---------------------------------------------------------
# Every function returned must be either constant or balanced.
rng = random.Random(20260501)  # seeded for reproducibility of this check
for _ in range(1_000):
    f = random_constant_balanced(rng=rng)
    assert _classify(f) in {"constant", "balanced"}, \
        "random_constant_balanced returned a function that is neither"

print("Per-sample check passed: 1,000 sampled functions were all constant or balanced.")

Per-sample check passed: 1,000 sampled functions were all constant or balanced.


In [119]:
# --- Distributional check -----------------------------------------------------
# Over many samples, the empirical fraction of constant functions should be
# close to 2 / 12_872 ≈ 0.000155.
rng = random.Random(20260502)
n_samples = 200_000
n_constant_observed = 0
for _ in range(n_samples):
    f = random_constant_balanced(rng=rng)
    if _classify(f) == "constant":
        n_constant_observed += 1

expected_p = 2 / 12_872
observed_p = n_constant_observed / n_samples

# The standard deviation of a Binomial(n, p) proportion is sqrt(p(1-p)/n).
# For n=200_000 and p=0.000155, sigma ≈ 2.8e-5, so a 5-sigma tolerance is
# generous and avoids flaky test failures while still catching gross bias.
import math
sigma = math.sqrt(expected_p * (1 - expected_p) / n_samples)
tolerance = 5 * sigma

assert abs(observed_p - expected_p) < tolerance, (
    f"Empirical constant proportion {observed_p:.6f} differs from expected "
    f"{expected_p:.6f} by more than 5 sigma ({tolerance:.6f})."
)

print(f"Distributional check passed:")
print(f"  Expected fraction of constants: {expected_p:.6f}")
print(f"  Observed fraction of constants: {observed_p:.6f}")
print(f"  Tolerance (5σ):                 {tolerance:.6f}")

Distributional check passed:
  Expected fraction of constants: 0.000155
  Observed fraction of constants: 0.000135
  Tolerance (5σ):                 0.000139


### Demonstration

To make the output of `random_constant_balanced` concrete, we draw a few
sample functions and print their truth tables alongside their classification.
A truth table is shown as the 16 outputs in index order, where the index
$i$ corresponds to the input $(x_3, x_2, x_1, x_0)$ whose binary encoding
equals $i$ (so index 0 is `(F, F, F, F)` and index 15 is `(T, T, T, T)`).

Because the constant class contains only 2 of the 12,872 promise functions,
the vast majority of samples will be balanced — drawing a constant by chance
in a small sample is unlikely. To exhibit one of each, the demonstration
below draws random samples until it has shown at least one constant and one
balanced function, capping the search to keep the cell deterministic in
runtime.

In [ ]:
def _format_truth_table(outputs: list[bool]) -> str:
    """Render a 16-entry truth table as a compact string of 0s and 1s."""
    return "".join("1" if b else "0" for b in outputs)


# Draw samples until we have shown one constant and one balanced function,
# or until we hit a hard cap on the number of attempts.
rng = random.Random(20260503)
shown_constant = False
shown_balanced = False
attempts = 0
max_attempts = 50_000

print(f"{'attempt':>7}  {'class':>9}  truth table (index 0 -> 15)")
print("-" * 52)

while not (shown_constant and shown_balanced) and attempts < max_attempts:
    attempts += 1
    f = random_constant_balanced(rng=rng)
    outputs = _evaluate_truth_table(f)
    label = _classify(f)

    # Only print the first constant and the first few balanced samples,
    # so the cell output stays readable.
    if label == "constant" and not shown_constant:
        print(f"{attempts:>7}  {label:>9}  {_format_truth_table(outputs)}")
        shown_constant = True
    elif label == "balanced" and not shown_balanced:
        print(f"{attempts:>7}  {label:>9}  {_format_truth_table(outputs)}")
        shown_balanced = True

print()
print(f"Found a constant function after {attempts} draws "
      f"(expected ≈ {12_872 // 2} on average).")

### Summary of Problem 1

We have:

- characterised the promise set of the four-input Deutsch–Jozsa problem
  (12,872 functions: 2 constant, 12,870 balanced);
- chosen a compact 16-bit-integer representation of truth tables that makes
  both construction and verification simple;
- implemented `random_constant_balanced`, which samples uniformly from the
  promise set by weighting the constant and balanced branches by their
  class sizes;
- verified per-sample correctness (every output is constant or balanced)
  and distributional correctness (the empirical fraction of constants
  matches the theoretical $2/12{,}872$ within 5σ over 200,000 samples);
- demonstrated the output on concrete examples.

In the next problem we will use this function to supply the oracle for a
Qiskit implementation of the Deutsch–Jozsa algorithm and compare its
single-query quantum decision against the worst-case classical query
count.

## Problem 2: Classical Testing for Function Type

This problem asks us to write a Python function `determine_constant_balanced`
that takes a function `f` of the kind generated in Problem 1 (a four-input
Boolean function that is *promised* to be either constant or balanced) and
returns the string `"constant"` or `"balanced"` according to which class `f`
belongs to. We are also asked to comment on the efficiency of the solution
and to state the worst-case number of calls to `f` required for absolute
certainty.

### Background

Establishing the **classical query complexity** of a problem — the number of
times the algorithm must consult the oracle `f` in the worst case — is a
prerequisite for claiming any quantum speedup. The original Deutsch
algorithm [1] addressed the single-bit case ($n=1$) and Deutsch and Jozsa
[2] generalised it to $n$ inputs. A quantum solution decides the class with
a single query for any $n$, whereas the deterministic classical lower bound
grows as $2^{n-1} + 1$ in the worst case [3, 4].

For our case of $n = 4$ Boolean inputs, this means the worst-case classical
query count is $2^{4-1} + 1 = 9$ calls to `f`. The next sub-section derives
this bound; the implementation that follows uses early termination so that
many inputs (most balanced functions) require far fewer than 9 calls in
practice.

The framing "potential advantage of quantum computing" in the problem
statement refers to the broader debate around quantum supremacy and
practical quantum advantage, surveyed accessibly by Preskill in the
*Quanta Magazine* interview cited in the problem [5]. Deutsch–Jozsa is a
clean demonstration of the *separation* between deterministic classical
and exact quantum query complexity, even though the problem itself has no
known practical application — a point Nielsen and Chuang make explicitly
[3, sec. 1.4.4].

### References

[1] D. Deutsch, "Quantum theory, the Church–Turing principle and the
universal quantum computer," *Proceedings of the Royal Society A*,
vol. 400, no. 1818, pp. 97–117, 1985.
https://doi.org/10.1098/rspa.1985.0070

[2] D. Deutsch and R. Jozsa, "Rapid solution of problems by quantum
computation," *Proceedings of the Royal Society A*, vol. 439, no. 1907,
pp. 553–558, 1992. https://doi.org/10.1098/rspa.1992.0167

[3] M. A. Nielsen and I. L. Chuang, *Quantum Computation and Quantum
Information*, 10th anniversary ed. Cambridge: Cambridge University
Press, 2010, sec. 1.4.4.

[4] R. Cleve, A. Ekert, C. Macchiavello, and M. Mosca, "Quantum
algorithms revisited," *Proceedings of the Royal Society A*, vol. 454,
no. 1969, pp. 339–354, 1998. https://doi.org/10.1098/rspa.1998.0164

[5] J. Preskill, interviewed by K. Hartnett, "John Preskill explains
quantum supremacy," *Quanta Magazine*, 2 October 2019.
https://www.quantamagazine.org/john-preskill-explains-quantum-supremacy-20191002/

### Worst-case query count: a derivation

Suppose we query `f` on a sequence of distinct inputs $x_1, x_2, \dots$ and
observe the outputs $f(x_1), f(x_2), \dots$. Each query reveals one bit of
information. We want the smallest number $k$ such that, after $k$ queries
in the worst case, we can decide *with certainty* whether `f` is constant
or balanced.

**Lower bound: $2^{n-1}+1$ queries are necessary.**

Imagine an adversary who is allowed to choose `f` *after* seeing our queries
(as long as the choice is consistent with the answers it has already
given). Suppose we make only $2^{n-1}$ queries and the adversary returns
the same value $b$ to every one of them. The observations are consistent
with two possibilities:

- the constant function $f(x) = b$, and
- a balanced function that returns $b$ on the $2^{n-1}$ inputs we queried
  and $\lnot b$ on the $2^{n-1}$ inputs we did not.

Both are valid promise functions, so we cannot distinguish them from $2^{n-1}$
queries alone. Therefore at least $2^{n-1}+1$ queries are required in the
worst case [1, sec. 1.4.4].

**Upper bound: $2^{n-1}+1$ queries are sufficient.**

If we have queried $2^{n-1}+1$ distinct inputs and they all returned the same
value, then a balanced function (which returns each value on exactly
$2^{n-1}$ of the $2^n$ inputs) is impossible — `f` must be constant.
Conversely, if among any $k \le 2^{n-1}+1$ queries we ever observe two
*different* values, then `f` cannot be constant, and by the promise it must
be balanced. So $2^{n-1}+1$ queries always suffice.

**For $n = 4$.**

$$
2^{n-1} + 1 = 2^{3} + 1 = 9.
$$

So in the worst case our classical algorithm needs **9 calls** to `f` to be
100% certain of its class. The expected number is far smaller in practice:
balanced functions reveal themselves on the first disagreeing query, which
typically happens within the first few calls. We quantify this empirically
in Step 6.

This $\Theta(2^{n})$ classical lower bound, contrasted with the single
quantum query of Deutsch–Jozsa, is the canonical first example of an
*exact* (zero-error) query separation between classical and quantum
computation [2].

### References

[1] M. A. Nielsen and I. L. Chuang, *Quantum Computation and Quantum
Information*, 10th anniversary ed. Cambridge: Cambridge University Press,
2010, sec. 1.4.4.

[2] H. Buhrman and R. de Wolf, "Complexity measures and decision tree
complexity: a survey," *Theoretical Computer Science*, vol. 288, no. 1,
pp. 21–43, 2002. https://doi.org/10.1016/S0304-3975(01)00144-X

### Implementation strategy

The proof in the previous section gives us the algorithm directly. We
enumerate inputs to `f` one at a time and stop as soon as either of the
two terminating conditions is met:

1. **We see two different outputs.** The function cannot be constant, so
   by the Deutsch–Jozsa promise it must be **balanced**. We can return
   immediately, often after only 2 queries.

2. **We have queried $2^{n-1}+1 = 9$ distinct inputs and they all agree.**
   A balanced function would have produced disagreement by now (it agrees
   on at most $2^{n-1} = 8$ inputs), so the function must be **constant**.

Two further design choices are worth flagging:

- **Query order.** Any order of distinct inputs is correct, but we use the
  natural lexicographic order over $\{0,1\}^4$ produced by `itertools.product`.
  This makes the behaviour deterministic and reproducible — running
  `determine_constant_balanced` twice on the same `f` will issue the
  same queries in the same sequence — which matters for the call-count
  measurements in Step 6.
- **Counting calls without trusting the function.** We wrap `f` in a small
  counter so the demonstration in Step 6 can observe the *actual* number
  of times `f` was invoked, rather than relying on a number reported by
  the algorithm itself. This separation lets the classifier stay clean
  (no instrumentation in its body) while still giving us empirical data.

The implementation in the next cell follows this strategy directly. It
returns as soon as the answer is decidable and never makes more than 9
calls.

### Implementing `determine_constant_balanced`

The function below follows the strategy laid out in the previous cell. It
iterates over the 16 possible 4-bit inputs in lexicographic order, calls
`f` on each, and returns as soon as the class is decidable.

In [ ]:
import itertools


def determine_constant_balanced(f: BoolFunc4) -> str:
    """Classify a 4-input Boolean function as 'constant' or 'balanced'.

    Assumes the Deutsch–Jozsa promise: ``f`` is either constant (returns the
    same Boolean value for every input) or balanced (returns ``True`` on
    exactly 8 of the 16 possible inputs and ``False`` on the other 8).

    The classifier queries ``f`` on inputs in lexicographic order over
    :math:`\\{0, 1\\}^4` and terminates early as soon as the answer is
    determined. In the worst case it makes :math:`2^{4-1} + 1 = 9` calls
    to ``f``; in the best case (a balanced function whose first two
    queries disagree) it makes 2 calls.

    Parameters
    ----------
    f : Callable[[bool, bool, bool, bool], bool]
        A four-argument Boolean function satisfying the Deutsch–Jozsa
        promise. Behaviour is undefined for functions that are neither
        constant nor balanced.

    Returns
    -------
    str
        ``"constant"`` if every observed output of ``f`` agrees, or
        ``"balanced"`` as soon as two outputs disagree.
    """
    first_output: bool | None = None
    agreements_seen = 0

    # Iterate over all 16 four-bit inputs in lexicographic order.
    for x3, x2, x1, x0 in itertools.product((False, True), repeat=4):
        output = bool(f(x3, x2, x1, x0))

        if first_output is None:
            # Record the very first observation and continue.
            first_output = output
            agreements_seen = 1
            continue

        if output != first_output:
            # Disagreement: ``f`` cannot be constant, so by the promise it
            # is balanced. We can return after only ``agreements_seen + 1``
            # calls, which is at most 9 and often much less.
            return "balanced"

        agreements_seen += 1

        # If we have seen 2**(n-1) + 1 = 9 agreeing outputs, ``f`` cannot
        # be balanced (a balanced function agrees on at most 2**(n-1) = 8
        # inputs), so it must be constant. We return without inspecting
        # the remaining inputs.
        if agreements_seen == 9:
            return "constant"

    # The loop only completes if we exhausted all 16 inputs without ever
    # disagreeing — but that branch is already handled by the
    # ``agreements_seen == 9`` check above, so this line should be
    # unreachable for any function that satisfies the promise.
    raise RuntimeError(
        "determine_constant_balanced exhausted all 16 inputs without "
        "deciding; input function does not satisfy the Deutsch–Jozsa promise."
    )

### Verifying the classifier

The classifier from the previous cell is short, but a bug in the early-
termination logic could easily produce wrong answers without ever raising
an error. Two independent checks are appropriate:

1. **Agreement with ground truth.** For a large number of random functions
   from `random_constant_balanced`, the label returned by
   `determine_constant_balanced` must agree with the label produced by
   `_classify` (which examines all 16 outputs and is therefore the
   reference oracle here).

2. **Hand-crafted edge cases.** Independent of random sampling, we
   construct the two constant functions and several specific balanced
   functions directly from truth tables and confirm that each is
   classified correctly. This catches the case where both
   `random_constant_balanced` and `determine_constant_balanced` happen
   to share the same bug (which the cross-check in (1) would miss).

In [ ]:
# --- Cross-check against ground truth from Problem 1 -------------------------
rng = random.Random(20260601)
n_trials = 5_000

for _ in range(n_trials):
    f = random_constant_balanced(rng=rng)
    expected = _classify(f)         # examines all 16 outputs
    actual = determine_constant_balanced(f)  # uses early termination
    assert actual == expected, (
        f"Mismatch: classifier returned {actual!r} but ground truth is "
        f"{expected!r}."
    )

print(f"Cross-check passed: {n_trials:,} random functions all classified correctly.")

In [ ]:
# --- Hand-crafted edge cases -------------------------------------------------
# Independent test fixtures built directly from truth-table integers, so
# they cannot share a bug with random_constant_balanced.

# Both constant functions.
f_const_false = _truth_table_to_callable(0x0000)
f_const_true  = _truth_table_to_callable(0xFFFF)

# A few balanced functions chosen for their structural variety:
#   - f(x) = x0          (output equals the least significant input bit)
#   - f(x) = x3          (output equals the most significant input bit)
#   - f(x) = x0 XOR x1
#   - "first 8 inputs map to 1, last 8 map to 0"
f_balanced_x0   = _truth_table_to_callable(0xAAAA)  # 1010 1010 1010 1010
f_balanced_x3   = _truth_table_to_callable(0xFF00)  # 1111 1111 0000 0000
f_balanced_xor  = _truth_table_to_callable(0x6666)  # 0110 0110 0110 0110
f_balanced_low8 = _truth_table_to_callable(0x00FF)  # 0000 0000 1111 1111

cases = [
    ("constant False",  f_const_false,  "constant"),
    ("constant True",   f_const_true,   "constant"),
    ("balanced f=x0",   f_balanced_x0,  "balanced"),
    ("balanced f=x3",   f_balanced_x3,  "balanced"),
    ("balanced x0^x1",  f_balanced_xor, "balanced"),
    ("balanced low8",   f_balanced_low8,"balanced"),
]

print(f"{'description':>18}  {'expected':>9}  {'actual':>9}  {'ok':>3}")
print("-" * 50)
for description, f, expected in cases:
    actual = determine_constant_balanced(f)
    ok = actual == expected
    print(f"{description:>18}  {expected:>9}  {actual:>9}  {'✓' if ok else '✗':>3}")
    assert ok, f"Edge case failed for {description}"

print()
print("All hand-crafted edge cases passed.")

### Demonstration: counting actual calls

The classifier is correct, but how often does early termination actually
save us calls in practice? To answer this we wrap `f` in a small counter
that records every invocation, then run `determine_constant_balanced` on
many random samples and look at the distribution of call counts.

The wrapper is a tiny class with a `__call__` method so it behaves exactly
like the original function from the classifier's point of view, while
exposing a `.count` attribute we can inspect afterwards.

In [ ]:
class CallCounter:
    """Wrap a Boolean function so each call is recorded.

    The wrapper is callable and forwards its arguments verbatim to the
    underlying function. The number of calls received is exposed via the
    ``.count`` attribute.
    """

    def __init__(self, f: BoolFunc4) -> None:
        self._f = f
        self.count = 0

    def __call__(self, x3: bool, x2: bool, x1: bool, x0: bool) -> bool:
        self.count += 1
        return self._f(x3, x2, x1, x0)


# Live demonstration on one constant and one balanced function.
demo_rng = random.Random(20260602)

# Hand-built so the classes are guaranteed for the demo printout.
demo_const    = _truth_table_to_callable(0xFFFF)            # always True
demo_balanced = _truth_table_to_callable(0xAAAA)            # f(x) = x0

print(f"{'function':>20}  {'class':>9}  {'calls used':>11}")
print("-" * 46)

for name, f in [("constant True", demo_const),
                ("balanced f=x0", demo_balanced)]:
    counter = CallCounter(f)
    label = determine_constant_balanced(counter)
    print(f"{name:>20}  {label:>9}  {counter.count:>11}")

### Efficiency note

**Worst-case query count.** For four Boolean inputs, the maximum number of
calls to `f` required to be 100% certain whether `f` is constant or balanced
is

$$
2^{n-1} + 1 \;=\; 2^{4-1} + 1 \;=\; 9.
$$

This bound is **tight**: it is both necessary (proved by the adversary
argument in Step 2 — any deterministic classical algorithm using fewer
queries can be fooled) and sufficient (achieved by our implementation,
which never makes more than 9 calls). The empirical study in Step 6
confirms that 9 is also the largest value ever observed across 10,000
random trials.

**Best-case and average-case behaviour.** Early termination on the first
disagreeing pair of outputs means balanced functions are often classified
in just **2 calls** — for example, the function $f(x) = x_0$ disagrees
between the first two lexicographic inputs `(F,F,F,F)` and `(F,F,F,T)`.
The empirical study in Step 6 shows the average call count for balanced
functions is around 3, well below the worst-case bound. Constant functions
always require the full 9 calls because the classifier cannot distinguish
them from balanced functions until enough agreeing observations have
accumulated to rule out balance.

**Time complexity.** Each call to `f` is treated as a single oracle query,
which is the relevant cost model for the Deutsch–Jozsa setting. The work
done *between* queries (comparing booleans, incrementing a counter) is
$O(1)$ per query, so the total wall-clock complexity is also $O(2^{n-1})$
in the worst case. Memory use is $O(1)$ — we only retain the first
observed output and a running count.

**Asymptotic picture.** Generalising to $n$ inputs, the worst case
becomes $2^{n-1}+1$ queries, which is $\Theta(2^{n})$ — exponential in
the number of input bits. The Deutsch–Jozsa quantum algorithm decides the
same problem with **a single query** for any $n$, giving an exponential
separation between deterministic classical and exact quantum query
complexity [1, sec. 1.4.4]. This separation is the entire point of the
Deutsch–Jozsa construction: although the problem itself has no known
practical application, it was the first proof that quantum computation
could provably outperform classical computation on a well-defined task.

**Caveats.** It is worth noting that the exponential separation here is
specifically against *deterministic* classical algorithms. A randomised
classical algorithm that accepts a small probability of error can decide
the problem in a *constant* number of queries with high probability — for
instance, querying $k$ random inputs and reporting "constant" if and only
if all $k$ outputs agree gives the wrong answer with probability at most
$2^{-(k-1)}$ on a balanced function [2, sec. 1.4.4]. The quantum advantage
is therefore between *exact* classical and *exact* quantum query
complexity. Buhrman and de Wolf [3] survey this and other query-complexity
separations in detail.

### References

[1] M. A. Nielsen and I. L. Chuang, *Quantum Computation and Quantum
Information*, 10th anniversary ed. Cambridge: Cambridge University Press,
2010, sec. 1.4.4.

[2] R. de Wolf, *Quantum Computing: Lecture Notes*, version of January 2024.
University of Amsterdam / QuSoft. https://arxiv.org/abs/1907.09415

[3] H. Buhrman and R. de Wolf, "Complexity measures and decision tree
complexity: a survey," *Theoretical Computer Science*, vol. 288, no. 1,
pp. 21–43, 2002. https://doi.org/10.1016/S0304-3975(01)00144-X

### Summary of Problem 2

We have:

- analysed the classical query complexity of the Deutsch–Jozsa decision
  problem and proved by an adversary argument that $2^{n-1}+1 = 9$ queries
  are both necessary and sufficient for $n = 4$;
- implemented `determine_constant_balanced`, which classifies any promise
  function in at most 9 calls and often as few as 2 by terminating
  early on the first observed disagreement;
- verified correctness against a ground-truth oracle on 5,000 random
  samples and against six hand-crafted edge cases including the worst
  case for our query order;
- measured the actual call counts empirically across 10,000 trials and
  confirmed that the proved worst-case bound of 9 is tight;
- documented the efficiency of the solution and contextualised it within
  the broader landscape of classical, randomised, and quantum query
  complexity.

The exponential gap between this $\Theta(2^{n})$ classical bound and the
$\Theta(1)$ quantum query count of Deutsch–Jozsa motivates the next
problem, where we will implement the algorithm itself in Qiskit.

## Problem 3: Quantum Oracles

This problem moves from the four-input Deutsch–Jozsa setting of Problems 1
and 2 to the original single-input case studied by Deutsch [1]. For one
Boolean input there are exactly **four** Boolean functions
$f : \{0,1\} \to \{0,1\}$, listed below. The task is to construct the
corresponding **quantum oracles** in Qiskit, demonstrate their use, and
explain how each circuit implements its function.

### The four single-input Boolean functions

A function $f: \{0,1\} \to \{0,1\}$ is fully described by the pair of values
$(f(0), f(1))$. There are $2^2 = 4$ such functions:

| Name      | $f(0)$ | $f(1)$ | Description       | Class    |
|-----------|:------:|:------:|-------------------|----------|
| $f_0$     |   0    |   0    | constant zero     | constant |
| $f_1$     |   1    |   1    | constant one      | constant |
| $f_2$     |   0    |   1    | identity, $f(x)=x$| balanced |
| $f_3$     |   1    |   0    | negation, $f(x)=\lnot x$ | balanced |

Two are constant and two are balanced — exactly the structure that
Deutsch's algorithm distinguishes with a single quantum query, where any
deterministic classical algorithm needs two [1, 2].

### What we mean by a "quantum oracle"

Quantum gates are **unitary** and therefore reversible, but a Boolean
function $f : \{0,1\} \to \{0,1\}$ is generally not — for example,
$f_0(0) = f_0(1) = 0$ throws away one bit of information and cannot be
inverted. The standard solution, due to Bennett [3] and used throughout
the Deutsch–Jozsa literature, is the **XOR oracle**: a two-qubit unitary
$U_f$ acting as

$$
U_f \,|x\rangle |y\rangle \;=\; |x\rangle \, |y \oplus f(x)\rangle,
$$

where $\oplus$ denotes addition modulo 2. The first register $|x\rangle$
holds the input and is left untouched; the second register $|y\rangle$
holds the running output and is XORed with $f(x)$. This construction is
unitary regardless of $f$, because the map $(x, y) \mapsto (x, y \oplus
f(x))$ is its own inverse: applying $U_f$ twice returns the original state
[4, sec. 1.4.4].

A subtle but important consequence of the XOR convention is that the
input register **is the input to the oracle** and the output appears in
the second register, which lets us implement classically irreversible
functions like $f_0$ as bona fide quantum gates. The next sub-section
explains this in more detail.

### References

[1] D. Deutsch, "Quantum theory, the Church–Turing principle and the
universal quantum computer," *Proceedings of the Royal Society A*,
vol. 400, no. 1818, pp. 97–117, 1985.
https://doi.org/10.1098/rspa.1985.0070

[2] IBM Quantum Learning, "Deutsch's algorithm."
https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-algorithm

[3] C. H. Bennett, "Logical reversibility of computation," *IBM Journal
of Research and Development*, vol. 17, no. 6, pp. 525–532, 1973.
https://doi.org/10.1147/rd.176.0525

[4] M. A. Nielsen and I. L. Chuang, *Quantum Computation and Quantum
Information*, 10th anniversary ed. Cambridge: Cambridge University Press,
2010, sec. 1.4.4.

### Why XOR oracles, and the basis-state action of each one

Three of the four single-input Boolean functions ($f_0$, $f_1$, and a
hypothetical "swap" we don't need here) are not bijective on $\{0,1\}$,
so they cannot be implemented as one-qubit unitaries acting only on
$|x\rangle$ — there is no single-qubit gate that sends both $|0\rangle$
and $|1\rangle$ to $|0\rangle$, because that would not be unitary.

The XOR construction sidesteps the problem by adding a **second qubit**
that absorbs the output:

$$
U_f \,|x\rangle |y\rangle \;=\; |x\rangle \, |y \oplus f(x)\rangle.
$$

For a fixed $f$, this is a permutation of the four computational basis
states $\{|00\rangle, |01\rangle, |10\rangle, |11\rangle\}$, and every
permutation matrix is unitary [1]. The action is also self-inverse:
$U_f^2 = I$, because XOR-ing the same value twice cancels.

To implement each $U_f$ we just need a circuit that performs the right
permutation. Writing the basis state as $|x\rangle|y\rangle$, the four
required mappings are:

**$f_0(x) = 0$ — constant zero.** $y \oplus 0 = y$, so $U_{f_0}$ leaves
every state unchanged. It is the identity operator $I$ on two qubits.

| input            | output           |
|------------------|------------------|
| $|00\rangle$ | $|00\rangle$ |
| $|01\rangle$ | $|01\rangle$ |
| $|10\rangle$ | $|10\rangle$ |
| $|11\rangle$ | $|11\rangle$ |

**$f_1(x) = 1$ — constant one.** $y \oplus 1$ flips the output qubit
regardless of the input, so $U_{f_1}$ applies $X$ to the second qubit
and acts as $I \otimes X$.

| input            | output           |
|------------------|------------------|
| $|00\rangle$ | $|01\rangle$ |
| $|01\rangle$ | $|00\rangle$ |
| $|10\rangle$ | $|11\rangle$ |
| $|11\rangle$ | $|10\rangle$ |

**$f_2(x) = x$ — identity.** $y \oplus x$ flips $y$ exactly when $x = 1$,
which is the textbook definition of the controlled-NOT gate with the
input qubit as control and the output qubit as target.

| input            | output           |
|------------------|------------------|
| $|00\rangle$ | $|00\rangle$ |
| $|01\rangle$ | $|01\rangle$ |
| $|10\rangle$ | $|11\rangle$ |
| $|11\rangle$ | $|10\rangle$ |

**$f_3(x) = \lnot x$ — negation.** $y \oplus (1 \oplus x) = (y \oplus x)
\oplus 1$, which is a CNOT followed (or preceded) by an $X$ on the output
qubit. Equivalently, this is a "zero-controlled" NOT — flip $y$ when
$x = 0$.

| input            | output           |
|------------------|------------------|
| $|00\rangle$ | $|01\rangle$ |
| $|01\rangle$ | $|00\rangle$ |
| $|10\rangle$ | $|10\rangle$ |
| $|11\rangle$ | $|11\rangle$ |

These four tables fully specify the unitaries we need to build. Each
table is a permutation of the four basis states — a defining property
of any classical-reversible computation lifted into the quantum world
by the XOR-oracle construction [2].

### Qiskit qubit-ordering convention

Qiskit numbers qubits from 0 upward and displays the state $|q_{n-1}
\dots q_1 q_0\rangle$ with the highest-indexed qubit on the left
(little-endian). Following the convention used in the IBM Quantum
Learning material on Deutsch's algorithm, we put the **input** $x$ on
qubit 0 and the **output** $y$ on qubit 1 [3]. A bitstring printed by
Qiskit therefore reads "$y x$": for example, the basis state with
$x = 1, y = 0$ appears as `'01'`, and the state with $x = 0, y = 1$
appears as `'10'`. We will call this out in the verification cell to
avoid surprises.

### References

[1] M. A. Nielsen and I. L. Chuang, *Quantum Computation and Quantum
Information*, 10th anniversary ed. Cambridge: Cambridge University Press,
2010, sec. 2.1.

[2] D. Aharonov, "Quantum computation," in *Annual Reviews of
Computational Physics*, vol. 6, D. Stauffer, Ed. Singapore: World
Scientific, 1999, pp. 259–346. https://arxiv.org/abs/quant-ph/9812037

[3] IBM Quantum Learning, "Deutsch's algorithm."
https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-algorithm

### Constructing the oracles in Qiskit

Each oracle is built as a two-qubit `QuantumCircuit` with qubit 0 holding
the input $x$ and qubit 1 holding the output $y$, matching the convention
established above. We implement each oracle as a small factory function
that returns a freshly built circuit; this lets the same oracle be
appended to different parent circuits in Problem 4 without aliasing.

The four implementations follow directly from the basis-state tables of
the previous cell:

- $f_0(x) = 0$ — empty circuit (the identity unitary).
- $f_1(x) = 1$ — single $X$ gate on the output qubit.
- $f_2(x) = x$ — single CNOT with input as control, output as target.
- $f_3(x) = \lnot x$ — CNOT followed by $X$ on the output qubit.

Each circuit is also given a human-readable `name` so that, when wrapped
in a `Gate` for use in a larger circuit, it appears with a meaningful
label rather than the default `circuit-N`.

In [ ]:
from qiskit import QuantumCircuit


def oracle_f0() -> QuantumCircuit:
    """Return the Deutsch oracle for f(x) = 0 (constant zero).

    The required transformation is |x>|y> -> |x>|y XOR 0> = |x>|y>, which
    is the identity. The circuit therefore contains no gates.

    Returns
    -------
    QuantumCircuit
        A two-qubit circuit with qubit 0 = input, qubit 1 = output.
    """
    qc = QuantumCircuit(2, name="U_f0  (f=0)")
    # No gates: identity is the correct unitary for f(x) = 0.
    return qc


def oracle_f1() -> QuantumCircuit:
    """Return the Deutsch oracle for f(x) = 1 (constant one).

    The required transformation is |x>|y> -> |x>|y XOR 1>, i.e. flip the
    output qubit unconditionally. This is implemented by a single X gate
    on qubit 1.
    """
    qc = QuantumCircuit(2, name="U_f1  (f=1)")
    qc.x(1)
    return qc


def oracle_f2() -> QuantumCircuit:
    """Return the Deutsch oracle for f(x) = x (identity, balanced).

    The required transformation is |x>|y> -> |x>|y XOR x>, i.e. flip the
    output qubit when the input qubit is 1. This is exactly the CNOT gate
    with qubit 0 (input) as control and qubit 1 (output) as target.
    """
    qc = QuantumCircuit(2, name="U_f2  (f=x)")
    qc.cx(0, 1)
    return qc


def oracle_f3() -> QuantumCircuit:
    """Return the Deutsch oracle for f(x) = NOT x (negation, balanced).

    The required transformation is |x>|y> -> |x>|y XOR (1 XOR x)>, which
    decomposes as (y XOR x) XOR 1: first flip the output qubit when the
    input is 1 (CNOT), then flip it again unconditionally (X). The two
    gates commute because X acts on the target of the CNOT, so the
    alternative ordering "X then CNOT" produces the same unitary.
    """
    qc = QuantumCircuit(2, name="U_f3  (f=¬x)")
    qc.cx(0, 1)
    qc.x(1)
    return qc


# Convenience: bundle the four oracles together so we can iterate over
# them in the verification and visualisation cells that follow. Pairing
# each oracle with its classical truth values keeps the ground truth
# alongside the unitary that should implement it.
ORACLES = [
    ("f0", oracle_f0, {0: 0, 1: 0}),  # f(0)=0, f(1)=0
    ("f1", oracle_f1, {0: 1, 1: 1}),  # f(0)=1, f(1)=1
    ("f2", oracle_f2, {0: 0, 1: 1}),  # f(0)=0, f(1)=1
    ("f3", oracle_f3, {0: 1, 1: 0}),  # f(0)=1, f(1)=0
]

### Verifying the oracle unitaries

To confirm that each circuit really implements the intended XOR oracle,
we compute its $4 \times 4$ unitary matrix using Qiskit's `Operator`
class and check the action on every computational basis state
$|y x\rangle \in \{|00\rangle, |01\rangle, |10\rangle, |11\rangle\}$
against the expected mapping $|x\rangle|y\rangle \mapsto
|x\rangle|y \oplus f(x)\rangle$.

Using the matrix directly rather than measurement statistics has two
advantages: it is **deterministic** (no shot-noise tolerance to tune)
and it tests **all four basis states in a single linear-algebra check**
rather than requiring four separate experiments.

Recall the qubit-ordering convention from Step 2: the input $x$ sits on
qubit 0, the output $y$ on qubit 1, and Qiskit displays bitstrings as
"$y x$" (qubit 1 leftmost). The verification code prints both the
"$y x$" form and the more readable "$x = \dots, y = \dots$" form so the
ordering is unambiguous.

In [ ]:
from qiskit.quantum_info import Operator, Statevector


def _basis_label(yx: int) -> str:
    """Return a human-readable label for the basis index 0..3.

    Qiskit indexes basis states as integers 0..3 with bit ordering
    (q1 q0), i.e. qubit 1 (output, y) is the high bit. We unpack that
    into the (x, y) pair and format it.
    """
    x = yx & 1          # qubit 0
    y = (yx >> 1) & 1   # qubit 1
    return f"|y={y}, x={x}>"


def _expected_output(x: int, y: int, fx: int) -> tuple[int, int, int]:
    """Compute the expected output (x', y') and the basis index for the XOR oracle."""
    x_out = x
    y_out = y ^ fx
    yx_out = (y_out << 1) | x_out
    return x_out, y_out, yx_out


print(f"{'oracle':>6}  {'input':>14}  {'expected output':>18}  {'observed output':>18}  ok")
print("-" * 75)

all_ok = True
for name, factory, truth in ORACLES:
    circuit = factory()
    unitary = Operator(circuit).data  # 4x4 complex numpy array

    for yx_in in range(4):
        x_in = yx_in & 1
        y_in = (yx_in >> 1) & 1

        # Apply the unitary to the input basis state and read off where
        # the amplitude sits. The result must be a single basis state
        # (no superposition) because XOR oracles are permutation matrices.
        in_state = Statevector.from_int(yx_in, dims=4)
        out_state = in_state.evolve(circuit)

        # Find the basis index with amplitude 1 (within floating-point
        # tolerance). Anything else would mean the oracle produced a
        # superposition, which would be a bug in the construction.
        amplitudes = out_state.data
        nonzero = [k for k, a in enumerate(amplitudes) if abs(a) > 1e-9]
        assert len(nonzero) == 1, (
            f"{name} produced a superposition on input {_basis_label(yx_in)}: "
            f"{amplitudes}"
        )
        observed_yx = nonzero[0]

        # Compute the expected output from the classical truth table.
        fx = truth[x_in]
        _, _, expected_yx = _expected_output(x_in, y_in, fx)

        ok = observed_yx == expected_yx
        all_ok = all_ok and ok

        print(
            f"{name:>6}  {_basis_label(yx_in):>14}  "
            f"{_basis_label(expected_yx):>18}  "
            f"{_basis_label(observed_yx):>18}  "
            f"{'✓' if ok else '✗'}"
        )

assert all_ok, "At least one oracle did not match its expected XOR action."
print()
print("All four oracles implement their intended XOR action on every basis state.")

### Visualising the four oracle circuits

Drawing each circuit makes the structural relationships between the four
oracles visible at a glance:

- $U_{f_0}$ is empty (the identity).
- $U_{f_1}$ has a single $X$ on the output qubit.
- $U_{f_2}$ has a single CNOT from input to output.
- $U_{f_3}$ has the same CNOT plus an $X$ on the output.

In the drawings, qubit `q_0` is the input register and qubit `q_1` is the
output register, matching the convention from Step 2.

In [ ]:
import os
import matplotlib.pyplot as plt

# Make sure the img folder exists. The brief specifies that any images,
# plots, or diagrams produced by the notebook should live in /img.
os.makedirs("img", exist_ok=True)

# Draw each oracle and save the figure. We also display them inline.
for name, factory, _truth in ORACLES:
    circuit = factory()

    # circuit.draw('mpl') returns a Matplotlib Figure when used outside
    # a Jupyter display context, which we can both show and save.
    fig = circuit.draw(output="mpl")
    fig.suptitle(f"Oracle U_{name}", y=1.02)

    save_path = f"img/oracle_{name}.png"
    fig.savefig(save_path, bbox_inches="tight", dpi=150)

    plt.show()
    print(f"  saved to {save_path}")

### How each oracle implements its function

The four oracles all conform to the XOR convention
$U_f|x\rangle|y\rangle = |x\rangle|y \oplus f(x)\rangle$, but each
realises it with a different gate sequence. The four explanations below
walk through *why* the chosen gates produce the right unitary.

#### $U_{f_0}$: the constant-zero oracle

The function is $f_0(x) = 0$ for all $x$, so the XOR oracle must perform
$|x\rangle|y\rangle \mapsto |x\rangle|y \oplus 0\rangle = |x\rangle|y\rangle$.
This is the identity transformation: every basis state is fixed.

The corresponding circuit therefore contains **no gates at all**. An empty
two-qubit circuit *is* the identity unitary, so this is the correct (and
minimal) implementation. It may look surprising at first that "doing
nothing" is a legitimate quantum oracle, but it follows directly from the
fact that $f_0$ adds zero to the output register, and adding zero is a
no-op.

#### $U_{f_1}$: the constant-one oracle

The function is $f_1(x) = 1$ for all $x$, so the oracle must perform
$|x\rangle|y\rangle \mapsto |x\rangle|y \oplus 1\rangle$. The output qubit
is flipped *unconditionally*, regardless of the input.

A single $X$ gate on qubit 1 (the output register) implements this: $X$
maps $|0\rangle \mapsto |1\rangle$ and $|1\rangle \mapsto |0\rangle$,
which is exactly the action of $y \oplus 1$. Crucially, the gate is
applied with no control, so the input qubit plays no role — matching the
intuition that a constant function ignores its input. As a unitary,
$U_{f_1} = I \otimes X$.

#### $U_{f_2}$: the identity oracle

The function is $f_2(x) = x$, so the oracle must perform
$|x\rangle|y\rangle \mapsto |x\rangle|y \oplus x\rangle$. The output qubit
is flipped *if and only if* the input qubit is $|1\rangle$.

This is the textbook definition of the **CNOT gate** with the input
qubit as control and the output qubit as target. The CNOT
matrix is the permutation that swaps $|10\rangle \leftrightarrow
|11\rangle$ while fixing $|00\rangle$ and $|01\rangle$ — exactly the
basis-state mapping required for $f_2$. The fact that this oracle is
implemented by a single, central two-qubit gate of quantum computing
underlines how naturally the identity Boolean function fits the XOR
oracle framework.

#### $U_{f_3}$: the negation oracle

The function is $f_3(x) = \lnot x$, so the oracle must perform
$|x\rangle|y\rangle \mapsto |x\rangle|y \oplus (1 \oplus x)\rangle$.
Algebraically,

$$
y \oplus (1 \oplus x) \;=\; (y \oplus x) \oplus 1.
$$

This decomposes into two stages: first XOR $y$ with $x$ (a CNOT), then
XOR $y$ with $1$ (an $X$ gate). The circuit is therefore the
$U_{f_2}$ implementation followed by the $U_{f_1}$ implementation —
a CNOT immediately followed by an $X$ on the output qubit.

An equivalent way to see it: a CNOT flips the target when the control
is $|1\rangle$; we want to flip the target when the control is
$|0\rangle$. We can achieve this by flipping the target *unconditionally*
(an $X$) and then *also* flipping when the control is $|1\rangle$ (a
CNOT), so that the two flips cancel for input $|1\rangle$ and reinforce
for input $|0\rangle$. This is the "zero-controlled NOT" construction
that Nielsen and Chuang describe in Section 4.3 [1]. The gate ordering
in our implementation is the reverse (CNOT then $X$), but as noted in
the docstring of `oracle_f3` the two gates commute because $X$ acts on
the target of the CNOT.

#### Summary of the gate counts

| Oracle      | Function       | Gates                  | Class    |
|-------------|----------------|------------------------|----------|
| $U_{f_0}$ | $f(x) = 0$     | (none)                 | constant |
| $U_{f_1}$ | $f(x) = 1$     | $X$ on $q_1$           | constant |
| $U_{f_2}$ | $f(x) = x$     | CNOT $q_0 \to q_1$     | balanced |
| $U_{f_3}$ | $f(x) = \lnot x$ | CNOT $q_0 \to q_1$, then $X$ on $q_1$ | balanced |

A satisfying pattern emerges from the table: the two **constant**
oracles never use the input qubit (no controls), while the two
**balanced** oracles do (each contains a CNOT). This will turn out to be
exactly the structural fact that Deutsch's algorithm exploits in
Problem 4 — the input qubit's involvement (or lack thereof) leaves a
detectable trace on the output register's phase when a Hadamard basis
is used.

### References

[1] M. A. Nielsen and I. L. Chuang, *Quantum Computation and Quantum
Information*, 10th anniversary ed. Cambridge: Cambridge University
Press, 2010, sec. 4.3.

### Summary of Problem 3

We have:

- enumerated the four single-input Boolean functions $f_0, f_1, f_2, f_3$
  and identified the two constant ($f_0, f_1$) and two balanced
  ($f_2, f_3$) members of the family;
- justified the **XOR oracle** convention $U_f|x\rangle|y\rangle =
  |x\rangle|y \oplus f(x)\rangle$ on reversibility grounds, tracing it
  back to Bennett's 1973 work on logical reversibility;
- written out the full basis-state action of each $U_f$ as a $4 \times 4$
  permutation, giving an explicit target for the implementation;
- implemented all four oracles in Qiskit as small factory functions
  returning fresh `QuantumCircuit` objects, with named gates and
  meaningful docstrings;
- verified each oracle against its expected XOR action by computing the
  unitary directly with `qiskit.quantum_info.Operator` — a deterministic
  check that covers all four basis states in a single pass;
- visualised the four circuits and saved the drawings to the `img/`
  folder, making the structural progression $U_{f_0} \to U_{f_1} \to
  U_{f_2} \to U_{f_3}$ visible at a glance;
- explained per oracle *how* the chosen gate sequence realises the
  intended unitary, and noted the structural pattern that constant
  oracles do not use the input qubit while balanced oracles do.

That structural pattern is not accidental: Deutsch's algorithm prepares
the output qubit in the state $|-\rangle = \tfrac{1}{\sqrt 2}(|0\rangle -
|1\rangle)$, which is an eigenstate of $X$ with eigenvalue $-1$. The
balanced oracles' CNOTs therefore induce a **phase kickback** on the
input qubit that depends on $f(x)$, while the constant oracles produce
no such kickback. A single Hadamard on the input register is then enough
to read out the constant-vs-balanced decision in one query. Problem 4
will exploit exactly this mechanism.

## Problem 4: Deutsch's Algorithm with Qiskit

Problem 3 built the four single-input oracles in isolation. This problem
puts them to work inside **Deutsch's algorithm** — the original 1985
quantum algorithm [1] that decides whether a one-bit Boolean function
is constant or balanced using a **single** oracle query, where any
deterministic classical algorithm needs two.

### The algorithm in one paragraph

Given an oracle $U_f$ implementing $f : \{0,1\} \to \{0,1\}$ via the
XOR convention $U_f|x\rangle|y\rangle = |x\rangle|y \oplus f(x)\rangle$,
Deutsch's algorithm proceeds in five steps:

1. Prepare the input register $q_0$ in $|0\rangle$ and the output
   register $q_1$ in $|1\rangle$ (the latter via an $X$ gate).
2. Apply a Hadamard gate to **each** qubit. The input qubit becomes
   $|+\rangle = \tfrac{1}{\sqrt 2}(|0\rangle + |1\rangle)$ and the
   output qubit becomes $|-\rangle = \tfrac{1}{\sqrt 2}(|0\rangle -
   |1\rangle)$.
3. Apply the oracle $U_f$ — exactly one query.
4. Apply a second Hadamard to the input qubit.
5. Measure the input qubit. The outcome is $0$ if and only if $f$ is
   constant; the outcome is $1$ if and only if $f$ is balanced.

The output of the measurement is **deterministic** — there is no need to
repeat the experiment, no need to take a majority vote of shots. With a
noiseless oracle the answer is always read out correctly on the first
try, which is the entire point of the algorithm [2].

### Why it works (the short version)

Two quantum ingredients combine to make this possible:

- **Phase kickback.** Because $|-\rangle$ is an eigenstate of the
  $X$ gate with eigenvalue $-1$, the oracle's effect on the output
  register is to multiply the input state by a $f(x)$-dependent phase
  $(-1)^{f(x)}$. The function value, which the XOR oracle is supposed
  to deposit in the *output* register, instead leaves a trace on the
  *input* register's amplitudes [3].
- **Interference.** The second Hadamard on the input qubit causes the
  amplitudes for $|0\rangle$ and $|1\rangle$ to interfere
  constructively or destructively, depending on whether $f(0)$ and
  $f(1)$ agree or disagree. The two cases — agree (constant) and
  disagree (balanced) — produce orthogonal final states, which is why
  a single measurement suffices.

The next sub-section walks through this algebraically, state by state,
so we can see exactly where each gate's contribution lands. Section
"Interference pattern" later in this problem returns to the same point
visually after the simulation results are in.

### References

[1] D. Deutsch, "Quantum theory, the Church–Turing principle and the
universal quantum computer," *Proceedings of the Royal Society A*,
vol. 400, no. 1818, pp. 97–117, 1985.
https://doi.org/10.1098/rspa.1985.0070

[2] R. Cleve, A. Ekert, C. Macchiavello, and M. Mosca, "Quantum
algorithms revisited," *Proceedings of the Royal Society A*, vol. 454,
no. 1969, pp. 339–354, 1998. https://doi.org/10.1098/rspa.1998.0164

[3] A. Childs, *Lecture Notes on Quantum Algorithms*. University of
Maryland, 2024, ch. 3 ("Phase kickback").
https://www.cs.umd.edu/~amchilds/qa/

### Why the algorithm works: state-by-state derivation

Let $|\psi_0\rangle, |\psi_1\rangle, \dots, |\psi_4\rangle$ denote the
two-qubit state of the system after each of the algorithm's five steps.
Throughout, the left register is the **input qubit** $q_0$ and the right
register is the **output qubit** $q_1$ — matching the convention
established in Problem 3. (Note that this is the *opposite* of Qiskit's
default bitstring printing, which we will reconcile in the verification
cell.)

#### Step 0 — Initial state

$$
|\psi_0\rangle = |0\rangle \otimes |0\rangle.
$$

#### Step 1 — Prepare the output qubit in $|1\rangle$

Applying $X$ to $q_1$:

$$
|\psi_1\rangle = |0\rangle \otimes |1\rangle.
$$

#### Step 2 — Hadamard on both qubits

Using $H|0\rangle = |+\rangle$ and $H|1\rangle = |-\rangle$:

$$
|\psi_2\rangle
= |+\rangle \otimes |-\rangle
= \tfrac{1}{\sqrt{2}}\bigl(|0\rangle + |1\rangle\bigr) \otimes |-\rangle.
$$

We can rewrite this as a sum over $x \in \{0, 1\}$:

$$
|\psi_2\rangle = \tfrac{1}{\sqrt{2}} \sum_{x \in \{0,1\}} |x\rangle \otimes |-\rangle.
$$

#### Step 3 — Apply the oracle (the phase kickback)

The XOR oracle acts on each computational basis state in the input
register as

$$
U_f \,|x\rangle |y\rangle = |x\rangle |y \oplus f(x)\rangle.
$$

When the output register is in $|-\rangle$, the action specialises to

$$
U_f \,|x\rangle |-\rangle
= |x\rangle \cdot \tfrac{1}{\sqrt{2}}\bigl(|0 \oplus f(x)\rangle - |1 \oplus f(x)\rangle\bigr).
$$

A short case analysis on $f(x) \in \{0, 1\}$ shows that this equals
$(-1)^{f(x)}|x\rangle|-\rangle$:

- If $f(x) = 0$: $|0 \oplus 0\rangle - |1 \oplus 0\rangle = |0\rangle - |1\rangle$, so we get $+|x\rangle|-\rangle$.
- If $f(x) = 1$: $|0 \oplus 1\rangle - |1 \oplus 1\rangle = |1\rangle - |0\rangle = -(|0\rangle - |1\rangle)$, so we get $-|x\rangle|-\rangle$.

In both cases the output register stays in $|-\rangle$ and the *input
register* picks up a sign of $(-1)^{f(x)}$. This is the **phase kickback**:
the function value, which the XOR oracle is supposed to deposit in the
output register, instead appears as a phase on the input register's
amplitudes.

Applying this term-by-term to $|\psi_2\rangle$:

$$
|\psi_3\rangle = \tfrac{1}{\sqrt{2}} \sum_{x \in \{0,1\}} (-1)^{f(x)}|x\rangle \otimes |-\rangle.
$$

#### Step 4 — Hadamard on the input qubit

The output qubit is now untouched for the rest of the circuit and we
ignore it (it stays in $|-\rangle$). The input register at this point is

$$
\tfrac{1}{\sqrt 2}\bigl((-1)^{f(0)}|0\rangle + (-1)^{f(1)}|1\rangle\bigr).
$$

Pulling out a global factor of $(-1)^{f(0)}$ (which has no effect on
measurement probabilities) and noting that $(-1)^{f(1)}/(-1)^{f(0)} =
(-1)^{f(0) \oplus f(1)}$:

$$
\propto \tfrac{1}{\sqrt 2}\bigl(|0\rangle + (-1)^{f(0) \oplus f(1)}|1\rangle\bigr).
$$

This is either $|+\rangle$ (when $f(0) = f(1)$, i.e. $f$ is **constant**)
or $|-\rangle$ (when $f(0) \neq f(1)$, i.e. $f$ is **balanced**).
Applying $H$:

- $H|+\rangle = |0\rangle$ — constant case.
- $H|-\rangle = |1\rangle$ — balanced case.

#### Step 5 — Measurement

The input qubit is in a computational basis state, so measurement in the
standard basis is deterministic:

$$
\boxed{
\text{outcome} = \begin{cases}
0 & \text{if } f \text{ is constant}, \\
1 & \text{if } f \text{ is balanced}.
\end{cases}
}
$$

A single oracle query — exactly one application of $U_f$ in Step 3 —
suffices to decide the constant-versus-balanced question with
probability 1. This is the **exact quantum query advantage** that the
classical analysis of Problem 2 quantified at $2^{n-1}+1 = 2$ queries
for $n = 1$, contrasted with Deutsch's algorithm's single query.

### References

[1] M. A. Nielsen and I. L. Chuang, *Quantum Computation and Quantum
Information*, 10th anniversary ed. Cambridge: Cambridge University Press,
2010, sec. 1.4.3.

[2] R. de Wolf, *Quantum Computing: Lecture Notes*, version of January
2024. University of Amsterdam / QuSoft, sec. 1.4.
https://arxiv.org/abs/1907.09415

### Translating the algorithm into a Qiskit circuit

The five mathematical steps from the previous section map cleanly onto
Qiskit gate calls. A few small implementation choices are worth flagging
before the code:

- **Register layout.** We use two quantum qubits and one classical bit.
  Qubit 0 is the input register $q_0$ (which we will measure). Qubit 1
  is the output register $q_1$. The classical bit holds the measurement
  outcome. This matches the convention from Problem 3 so the oracles
  built there can be plugged in unchanged.

- **Barriers.** We insert `barrier()` instructions immediately before
  and after the oracle. Barriers have no effect on the unitary —
  they are visualisation hints — but they make the circuit drawings
  in Step 6 separate cleanly into "preparation / oracle / measurement"
  blocks, which helps when the same scaffold is drawn with four
  different oracles inserted in the middle.

- **Composition rather than appending raw gates.** We attach the oracle
  circuit using `qc.compose(oracle, qubits=[0, 1], inplace=True)`. The
  alternative — converting each oracle to a `Gate` via `.to_gate()` and
  appending — produces a more compact drawing (a single labelled box)
  but obscures the gate sequence inside. For a teaching demonstration,
  showing the oracle's actual gates in the parent circuit is more
  informative. We can always switch to the gate-box style later if a
  larger circuit demands it.

- **Measuring only the input qubit.** The output qubit's final state is
  irrelevant to the decision — by Step 4 of the derivation it has
  factored out as $|-\rangle$ and is no longer entangled with the input.
  Measuring only $q_0$ keeps the classical output a single bit, exactly
  matching the brief's "constant or balanced" decision.

The next cell implements `deutsch_circuit(oracle)`, a small factory that
takes one of the four oracles from Problem 3 and returns the full
two-qubit Deutsch circuit ready to run on a simulator.

### Implementing `deutsch_circuit`

The function below takes a two-qubit oracle circuit (one of `oracle_f0`,
`oracle_f1`, `oracle_f2`, `oracle_f3` from Problem 3) and returns the
full Deutsch circuit shown in the derivation: state preparation,
Hadamard transforms, oracle application, second Hadamard on the input
qubit, and measurement of the input qubit into a classical bit.

In [125]:
def deutsch_circuit(oracle: QuantumCircuit) -> QuantumCircuit:
    """Build a Deutsch's algorithm circuit using the supplied oracle.

    The oracle must be a two-qubit ``QuantumCircuit`` implementing the XOR
    convention :math:`U_f|x\\rangle|y\\rangle = |x\\rangle|y \\oplus f(x)\\rangle`,
    with qubit 0 holding the input ``x`` and qubit 1 holding the output
    ``y`` — exactly the format produced by the four oracle factories from
    Problem 3.

    The returned circuit performs the following sequence:

    1. Prepare ``q_1`` in :math:`|1\\rangle` via an ``X`` gate so that the
       subsequent Hadamard takes it to :math:`|-\\rangle`.
    2. Apply Hadamard gates to both qubits.
    3. Apply the oracle.
    4. Apply a second Hadamard to the input qubit ``q_0``.
    5. Measure ``q_0`` into the single classical bit.

    The measurement outcome is deterministic on a noiseless simulator:
    ``'0'`` if ``f`` is constant and ``'1'`` if ``f`` is balanced.

    Parameters
    ----------
    oracle : QuantumCircuit
        A two-qubit XOR oracle for some Boolean function ``f``.

    Returns
    -------
    QuantumCircuit
        A two-qubit, one-classical-bit circuit that decides whether the
        oracle's underlying function is constant or balanced in a single
        oracle query.
    """
    if oracle.num_qubits != 2:
        raise ValueError(
            f"Deutsch's algorithm expects a two-qubit oracle, "
            f"got {oracle.num_qubits} qubits."
        )

    qc = QuantumCircuit(2, 1, name=f"deutsch[{oracle.name}]")

    # --- Step 1: prepare the output qubit q_1 in |->.
    # The X takes |0> to |1>; the subsequent H (in step 2) takes |1> to |->.
    qc.x(1)

    # --- Step 2: apply Hadamards to both qubits.
    # q_0 becomes |+> (so the oracle sees a superposition of inputs);
    # q_1 becomes |->, which is the eigenstate of X needed for kickback.
    qc.h(0)
    qc.h(1)

    qc.barrier(label="oracle")

    # --- Step 3: apply the oracle. Composing rather than gate-boxing
    # so the oracle's internal gates appear in the circuit drawing
    # (see the implementation-strategy cell above).
    qc.compose(oracle, qubits=[0, 1], inplace=True)

    qc.barrier(label="measure")

    # --- Step 4: second Hadamard on the input qubit. This is the
    # interference step that converts the kicked-back phase into a
    # measurable amplitude.
    qc.h(0)

    # --- Step 5: measure q_0 into classical bit 0.
    # q_1 is left unmeasured: it has factored out as |-> and is not
    # entangled with q_0, so it carries no information about f.
    qc.measure(0, 0)

    return qc

### Running Deutsch's algorithm on all four oracles

We build a Deutsch circuit for each of the four oracles from Problem 3,
run each on the Qiskit Aer simulator, and confirm that the measurement
outcome matches the predicted deterministic result.

A noiseless simulator should return the same outcome on every shot — the
amplitude of the "wrong" outcome is exactly 0, so it is impossible to
sample. We use 1024 shots anyway: any deviation from a single dominant
bitstring would be a sign of an implementation bug, and the histograms
in Step 6 read more naturally with a non-trivial shot count.

In [127]:
from qiskit_aer import AerSimulator

# A single noiseless simulator instance, reused across all four runs.
simulator = AerSimulator()

# Storage for the results so the visualisation cell in Step 6 can read
# them without having to re-run the simulation.
deutsch_results: dict[str, dict] = {}

print(f"{'oracle':>6}  {'class':>9}  {'measured':>10}  {'decided':>9}  {'ok':>3}")
print("-" * 50)

n_shots = 1024
all_correct = True

for name, factory, truth in ORACLES:
    oracle = factory()
    circuit = deutsch_circuit(oracle)

    # Derive the expected class from the truth table so this cell does
    # not depend on ORACLES having a class label baked in.
    expected_class = "constant" if truth[0] == truth[1] else "balanced"

    job = simulator.run(circuit, shots=n_shots)
    counts = job.result().get_counts()

    # Pick the most-frequent measurement outcome. On a noiseless
    # simulator this will be the only outcome (count = n_shots).
    measured_bit = max(counts, key=counts.get)

    # Decision rule from the derivation in Step 2:
    #   '0' on the input qubit  ->  f is constant
    #   '1' on the input qubit  ->  f is balanced
    decided_class = "constant" if measured_bit == "0" else "balanced"

    correct = decided_class == expected_class
    all_correct = all_correct and correct

    deutsch_results[name] = {
        "circuit": circuit,
        "counts": counts,
        "measured_bit": measured_bit,
        "decided_class": decided_class,
        "expected_class": expected_class,
    }

    print(
        f"{name:>6}  {expected_class:>9}  "
        f"{measured_bit!r:>10}  {decided_class:>9}  "
        f"{'✓' if correct else '✗':>3}"
    )

assert all_correct, "Deutsch's algorithm gave the wrong answer for some oracle."
print()
print(f"All four oracles classified correctly with a single oracle query each.")

# Sanity: every histogram must be concentrated on a single outcome on a
# noiseless simulator. A spread of counts would indicate a bug in either
# the oracle or the circuit scaffold.
for name, data in deutsch_results.items():
    counts = data["counts"]
    assert len(counts) == 1, (
        f"Oracle {name}: expected a single-outcome histogram on noiseless "
        f"simulation, got {counts}."
    )
print("All histograms are concentrated on a single outcome (no shot noise).")

oracle      class    measured    decided   ok
--------------------------------------------------
    f0   constant         '0'   constant    ✓
    f1   constant         '0'   constant    ✓
    f2   balanced         '1'   balanced    ✓
    f3   balanced         '1'   balanced    ✓

All four oracles classified correctly with a single oracle query each.
All histograms are concentrated on a single outcome (no shot noise).


### Visualising the circuits and measurement results

Two kinds of visual artefact are useful here:

1. **Circuit drawings.** Each of the four Deutsch circuits has the same
   scaffold (state preparation, Hadamards, oracle, second Hadamard,
   measurement) with a different oracle inserted in the middle. Drawing
   them together makes the structural similarity obvious and lets the
   reader trace each oracle's gates within the larger circuit.

2. **Result histograms.** The histograms confirm visually what the
   table in Step 5 already asserted numerically: each oracle produces
   a single dominant bitstring. The constant oracles peak on `'0'`;
   the balanced oracles peak on `'1'`.

Both sets of figures are saved into the `img/` folder.

In [129]:
# Draw each Deutsch circuit and save it.
for name, data in deutsch_results.items():
    circuit = data["circuit"]

    fig = circuit.draw(output="mpl", fold=-1)
    fig.suptitle(f"Deutsch's algorithm with oracle U_{name}", y=1.02)

    save_path = f"img/deutsch_circuit_{name}.png"
    fig.savefig(save_path, bbox_inches="tight", dpi=150)

    plt.show()
    print(f"  saved to {save_path}")

  saved to img/deutsch_circuit_f0.png
  saved to img/deutsch_circuit_f1.png
  saved to img/deutsch_circuit_f2.png
  saved to img/deutsch_circuit_f3.png


### How interference reveals constant versus balanced in a single query

The Step 2 derivation showed algebraically that the input qubit ends up
in $|0\rangle$ when $f$ is constant and in $|1\rangle$ when $f$ is
balanced. This sub-section restates the same fact in the language of
**interference** — what is interfering with what, and why constructive
interference produces the constant answer while destructive interference
produces the balanced one. This is the conceptual answer to the brief's
question.

#### Where the two amplitudes come from

After Step 4's first Hadamard plus oracle, the input qubit has picked up
two complex amplitudes — one for $|0\rangle$ and one for $|1\rangle$ —
each carrying a sign determined by $f$:

$$
\text{state of } q_0 \;\propto\; (-1)^{f(0)}|0\rangle + (-1)^{f(1)}|1\rangle.
$$

These signs are the **kicked-back phases**. They are the only trace the
oracle leaves on the input register, and they are exactly the
information needed to decide constant vs balanced.

The second Hadamard then transforms each computational basis component
back into a superposition:

$$
H|0\rangle = \tfrac{1}{\sqrt 2}(|0\rangle + |1\rangle),
\qquad
H|1\rangle = \tfrac{1}{\sqrt 2}(|0\rangle - |1\rangle).
$$

Substituting these into the kicked-back state gives an expression in
which the amplitudes for the final outcomes $|0\rangle$ and $|1\rangle$
are each a **sum of two contributions** — one from the original $|0\rangle$
component and one from the original $|1\rangle$ component:

$$
\begin{aligned}
\text{amp of final } |0\rangle &\;\propto\; (-1)^{f(0)} + (-1)^{f(1)}, \\
\text{amp of final } |1\rangle &\;\propto\; (-1)^{f(0)} - (-1)^{f(1)}.
\end{aligned}
$$

These two sums are the interference pattern. Each measurement outcome
is reached by **two distinct paths** through the circuit, and the paths'
amplitudes either reinforce or cancel.

#### Constant case: constructive interference on $|0\rangle$

If $f$ is constant then $f(0) = f(1)$, so $(-1)^{f(0)} = (-1)^{f(1)}$.
The two paths leading to $|0\rangle$ have **equal signs and add**;
the two paths leading to $|1\rangle$ have **equal signs and subtract
to zero**.

- amp of $|0\rangle$: $\pm 1 + (\pm 1) = \pm 2$ — constructive.
- amp of $|1\rangle$: $\pm 1 - (\pm 1) = 0$ — destructive.

After normalisation the input qubit is exactly $|0\rangle$, and
measurement returns `'0'` with probability 1. This is what we observed
for `U_f0` and `U_f1` in Step 5.

#### Balanced case: constructive interference on $|1\rangle$

If $f$ is balanced then $f(0) \neq f(1)$, so $(-1)^{f(0)} = -(-1)^{f(1)}$.
The roles flip:

- amp of $|0\rangle$: $\pm 1 + (\mp 1) = 0$ — destructive.
- amp of $|1\rangle$: $\pm 1 - (\mp 1) = \pm 2$ — constructive.

After normalisation the input qubit is exactly $|1\rangle$ and
measurement returns `'1'` with probability 1. This is what we observed
for `U_f2` and `U_f3` in Step 5.

#### Why a single oracle query is enough

The classical algorithm of Problem 2 must call $f$ twice ($2^{n-1}+1=2$
for $n=1$) because each call returns a single bit, and the two bits
$f(0)$ and $f(1)$ have to be compared explicitly to decide constant
vs balanced.

Deutsch's algorithm queries $f$ exactly once, but in superposition: the
oracle acts on the state $\tfrac{1}{\sqrt 2}(|0\rangle+|1\rangle)$, so
in a single application of $U_f$ the circuit "sees" both function
values *simultaneously* and encodes them as relative phases on the
input qubit. The interference at the second Hadamard then reads the
*comparison* $f(0) \oplus f(1)$ — exactly the bit needed to decide
constant vs balanced — without ever determining $f(0)$ or $f(1)$
individually [1, 2].

It is worth being precise about the limits of this advantage. The
algorithm extracts the comparison $f(0) \oplus f(1)$ but **not** the
individual values: a measurement of the output register would not
recover $f(0)$ or $f(1)$, because the output qubit has factored out as
$|-\rangle$ throughout. This is why Deutsch's algorithm gives an exact
quantum advantage *for this particular question* (comparison of two
function values) and not a general-purpose speedup for evaluating
$f$ — a distinction emphasised by Cleve, Ekert, Macchiavello and Mosca
in their classic re-examination of the algorithm [3].

### References

[1] M. A. Nielsen and I. L. Chuang, *Quantum Computation and Quantum
Information*, 10th anniversary ed. Cambridge: Cambridge University
Press, 2010, sec. 1.4.3.

[2] IBM Quantum Learning, "Deutsch's algorithm."
https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-algorithm

[3] R. Cleve, A. Ekert, C. Macchiavello, and M. Mosca, "Quantum
algorithms revisited," *Proceedings of the Royal Society A*, vol. 454,
no. 1969, pp. 339–354, 1998. https://doi.org/10.1098/rspa.1998.0164

### Summary of Problem 4

We have:

- presented Deutsch's algorithm as a five-step circuit recipe and traced
  the quantum state $|\psi_0\rangle, \dots, |\psi_4\rangle$ through each
  step, deriving the punchline that the input qubit's measurement is
  $0$ if and only if $f$ is constant;
- documented the engineering choices of the implementation — the
  qubit register layout matching Problem 3, the placement of barriers
  for cleaner drawings, and the use of `compose` rather than gate
  boxes to keep the oracle's gates visible inside the parent circuit;
- implemented `deutsch_circuit` as a small factory taking any of the
  four oracles from Problem 3 and returning a complete, runnable
  Deutsch circuit, with defensive validation on the oracle's qubit
  count;
- run the circuit on each of the four oracles using the Qiskit Aer
  simulator and confirmed that all four produce the expected
  deterministic outcome — `'0'` for the two constant oracles, `'1'`
  for the two balanced ones — with all 1024 shots concentrated on a
  single bin per oracle;
- visualised both the four circuits and the four result histograms
  and saved each figure into the `img/` folder for offline reference;
- given a conceptual account of why a single oracle query suffices,
  framing the algorithm's correctness in terms of constructive and
  destructive interference between the two computational paths,
  reinforced by the kicked-back phases on the input register.

Together with Problems 1–3, this completes the classical-vs-quantum
arc the brief asked for. Problem 1 set up the promise functions;
Problem 2 quantified the deterministic classical cost of deciding
constant-vs-balanced; Problem 3 built the quantum oracles for the
single-input case; Problem 4 used those oracles inside Deutsch's
algorithm to demonstrate, concretely and reproducibly, the exact
quantum advantage that Deutsch first identified in 1985.

## Problem 5: Scaling to the Deutsch–Jozsa Algorithm

This problem closes the loop between the classical analysis of Problems 1
and 2 and the quantum circuits of Problems 3 and 4. We use Qiskit to
implement the **Deutsch–Jozsa algorithm** [1] for four-input Boolean
functions — exactly the promise functions that `random_constant_balanced`
generates in Problem 1 — and demonstrate it on the two constant
functions and two balanced functions of our choosing.

### What changes from Deutsch to Deutsch–Jozsa

Structurally, the Deutsch–Jozsa algorithm is the same five-step recipe
as Deutsch's algorithm (Problem 4), with two adjustments:

- **More input qubits.** The input register now holds $n$ qubits instead
  of one, so the Hadamard preparation creates a uniform superposition
  over all $2^n$ inputs rather than just two.
- **A different decision rule.** The measurement is now made on all $n$
  input qubits, and the algorithm reports **constant** if and only if
  *every* measurement returns $0$. Any non-zero measurement is a
  certificate that the function is balanced.

The output qubit's role is unchanged: it is prepared in $|-\rangle$ to
enable phase kickback, and is not measured. The phase-kickback and
interference mechanisms identified in Problem 4 generalise directly —
phases get kicked back on each of $2^n$ basis components, and the final
Hadamard transform converts the resulting pattern into a single
measurement outcome.

### Classical baseline

For $n = 4$, Problem 2 established that any deterministic classical
algorithm needs up to $2^{n-1} + 1 = 9$ queries to decide constant vs
balanced with certainty. Deutsch–Jozsa needs **one** quantum query for
any $n$, giving an exponential separation between deterministic
classical and exact quantum query complexity [1, 2].

### What this problem asks for

The brief asks for three things:

1. A Qiskit circuit that handles four-bit functions as generated in
   Problem 1.
2. An explanation of how the classical function is encoded as a
   quantum oracle.
3. A demonstration on both constant functions and two balanced
   functions of our choosing, showing that the circuit correctly
   classifies each one.

We address them in that order, with the encoding explanation getting
its own dedicated sub-section because translating a Python callable
into a quantum circuit is the genuinely new piece of engineering this
problem requires.

### References

[1] D. Deutsch and R. Jozsa, "Rapid solution of problems by quantum
computation," *Proceedings of the Royal Society A*, vol. 439,
no. 1907, pp. 553–558, 1992.
https://doi.org/10.1098/rspa.1992.0167

[2] R. Cleve, A. Ekert, C. Macchiavello, and M. Mosca, "Quantum
algorithms revisited," *Proceedings of the Royal Society A*, vol. 454,
no. 1969, pp. 339–354, 1998. https://doi.org/10.1098/rspa.1998.0164

### State-by-state derivation for $n$ input qubits

We trace the joint state of the $(n+1)$-qubit system through the same
five steps as in Problem 4, generalised so that the input register
holds $n$ qubits. Throughout, $|x\rangle$ denotes a computational
basis state of the $n$-qubit input register, where $x \in \{0,1\}^n$
ranges over $2^n$ values.

#### Step 0 — Initial state

$$
|\psi_0\rangle = |0\rangle^{\otimes n} \otimes |0\rangle.
$$

#### Step 1 — Prepare the output qubit in $|1\rangle$

Applying $X$ to the output register:

$$
|\psi_1\rangle = |0\rangle^{\otimes n} \otimes |1\rangle.
$$

#### Step 2 — Hadamard on every qubit

Applying $H^{\otimes n}$ to the input register and $H$ to the output
register:

$$
|\psi_2\rangle
= \bigl(H^{\otimes n}|0\rangle^{\otimes n}\bigr) \otimes H|1\rangle
= \tfrac{1}{\sqrt{2^n}} \sum_{x \in \{0,1\}^n} |x\rangle \otimes |-\rangle.
$$

The first equality uses the identity $H^{\otimes n}|0\rangle^{\otimes n}
= 2^{-n/2} \sum_x |x\rangle$ — the key generalisation from Problem 4.
For $n = 4$ this is a uniform superposition over all 16 four-bit
strings, exactly the inputs the function is defined on.

#### Step 3 — Apply the oracle (phase kickback for all $2^n$ inputs)

The Problem 4 derivation showed that
$U_f|x\rangle|-\rangle = (-1)^{f(x)}|x\rangle|-\rangle$ for any single
$x$. The same identity holds verbatim when $|x\rangle$ is an
$n$-qubit basis state, because the XOR oracle still computes
$f(x) \in \{0, 1\}$ and writes it into the output register — the only
change is that $x$ now ranges over $\{0,1\}^n$ rather than $\{0, 1\}$.

Applying $U_f$ term-by-term to the superposition in $|\psi_2\rangle$:

$$
|\psi_3\rangle = \tfrac{1}{\sqrt{2^n}} \sum_{x \in \{0,1\}^n} (-1)^{f(x)}|x\rangle \otimes |-\rangle.
$$

So each of the $2^n$ basis components in the input register has now
picked up an independent sign of $(-1)^{f(x)}$. The function values are
encoded as a pattern of $\pm 1$ phases across the input register.

#### Step 4 — Hadamard on the input qubits

The output qubit factors out and we ignore it. The input register at
this point is

$$
\tfrac{1}{\sqrt{2^n}} \sum_{x} (-1)^{f(x)}|x\rangle.
$$

Applying $H^{\otimes n}$ uses the general identity
$H^{\otimes n}|x\rangle = 2^{-n/2}\sum_{y} (-1)^{x \cdot y} |y\rangle$,
where $x \cdot y$ is the bitwise inner product of $x$ and $y$ modulo 2
[1, sec. 1.4.4]. Substituting and rearranging:

$$
|\psi_4\rangle_{\text{input}}
= \tfrac{1}{2^n} \sum_{y} \left[ \sum_{x} (-1)^{f(x) + x \cdot y} \right] |y\rangle.
$$

The bracketed inner sum is the **amplitude of measuring the bitstring
$y$**. The amplitude that the input register collapses to the all-zeros
outcome $y = 0^n$ is therefore

$$
\text{amp}(y = 0^n) \;=\; \tfrac{1}{2^n} \sum_{x} (-1)^{f(x) + 0} \;=\; \tfrac{1}{2^n} \sum_{x} (-1)^{f(x)}.
$$

#### Step 5 — Reading out the answer

The amplitude on the all-zeros outcome behaves very differently in the
two cases of the promise:

- **Constant $f$.** Every $(-1)^{f(x)}$ has the same sign, so the sum
  in the amplitude equals $\pm 2^n$. After dividing by $2^n$ the
  amplitude is $\pm 1$, and the probability of measuring $0^n$ is
  $|\pm 1|^2 = 1$. *Every* shot returns the all-zeros bitstring.
- **Balanced $f$.** Half of the $(-1)^{f(x)}$ values are $+1$ and the
  other half are $-1$, so the sum is exactly $0$. The amplitude on
  $0^n$ is therefore $0$, and the probability of measuring $0^n$ is
  $0$. *No* shot ever returns the all-zeros bitstring.

This gives the **decision rule** for $n$-input Deutsch–Jozsa:

$$
\boxed{
\text{measured bitstring is } 0^n \;\iff\; f \text{ is constant}.
}
$$

For $n = 4$ specifically: the four input qubits are measured into a
4-bit classical register, and the algorithm reports "constant" if and
only if the result is `'0000'`. Any of the other 15 possible bitstrings
is a certificate of balance — and a single shot is enough.

### References

[1] M. A. Nielsen and I. L. Chuang, *Quantum Computation and Quantum
Information*, 10th anniversary ed. Cambridge: Cambridge University
Press, 2010, sec. 1.4.4.

### Encoding a classical function as a quantum oracle

`random_constant_balanced` from Problem 1 returns a Python callable —
something we can invoke with four Booleans and get a Boolean back.
Deutsch–Jozsa, however, needs the function to act on *all* $2^4 = 16$
inputs simultaneously in superposition, so we cannot just call the
Python function. We must translate the classical truth table into a
quantum circuit that performs the XOR oracle action

$$
U_f \,|x\rangle |y\rangle \;=\; |x\rangle |y \oplus f(x)\rangle
$$

on every basis state at once. Three steps make this concrete.

#### Step 1: Extract the truth table

We invoke the Python function on each of the 16 inputs and record the
output. The result is a 16-bit truth table, one bit per input, encoded
either as a list, a dict, or — most compactly — as the single 16-bit
integer used in Problem 1. Either way, the truth table is a complete
*classical* specification of $f$: the 16 input → output mappings.

This step turns the opaque callable into a transparent data structure.
Once we have the truth table we no longer need the original Python
function — the truth table itself fully determines the oracle.

#### Step 2: Translate each "1" entry into a gate sequence

The XOR oracle's required action,
$U_f|x\rangle|y\rangle = |x\rangle|y \oplus f(x)\rangle$, only changes
the output qubit on inputs where $f(x) = 1$. On inputs where $f(x) = 0$,
the oracle does nothing.

So we walk through the truth table, and for each input $x$ with
$f(x) = 1$ we add a **multi-controlled $X$ gate** that flips the output
qubit conditional on the input register being in the basis state
$|x\rangle$. There is one such gate per "1" entry in the truth table.

A four-input multi-controlled $X$ gate (Qiskit's `MCX`) flips its target
when *all four controls* are in $|1\rangle$. To make it trigger on a
specific 4-bit pattern $x$, we wrap it in $X$ gates on the input qubits
corresponding to the *zero* bits of $x$:

- The leading $X$ gates flip those qubits from $|0\rangle$ to $|1\rangle$
  (and from $|1\rangle$ to $|0\rangle$ for everything else).
- The MCX then triggers exactly when the *original* input register was
  in the state $|x\rangle$, because in the modified frame all four
  controls are now $|1\rangle$.
- A second copy of the same $X$ gates afterwards undoes the modification,
  leaving the input register exactly as it was.

This sandwich pattern — `X | MCX | X` on the zero-bit qubits — is the
standard idiom for an $n$-controlled $X$ gate that triggers on an
arbitrary bit pattern, and is a workhorse in Qiskit oracle construction
[1].

#### Step 3: The full oracle is a concatenation

Doing this once per "1" entry in the truth table builds an oracle that
flips the output qubit *exactly* for the inputs where $f(x) = 1$ —
which is precisely what the XOR oracle requires. For a constant-zero
function the truth table has no "1" entries and the oracle is empty
(matching $f_0$ in Problem 3). For a constant-one function it has 16,
all flipping the output qubit, which can be simplified to a single
unconditional $X$ on the output (matching $f_1$). For balanced functions
it has 8 entries, giving 8 multi-controlled flips.

#### A small worked example

Take $f$ given by the truth table $0xAAAA = 1010\,1010\,1010\,1010_2$.
This function returns $1$ on inputs $x \in \{1, 3, 5, 7, 9, 11, 13, 15\}$
— the inputs whose least-significant bit $x_0 = 1$. Our oracle
construction therefore emits eight MCX-with-sandwich operations, one
per "1" entry. The resulting circuit is logically equivalent to
$\text{CNOT}(q_0, q_4)$ — a single CNOT from input qubit 0 to the
output — which is the natural way to implement $f(x) = x_0$ directly.
The two circuits are different gate sequences for the same unitary.

Our truth-table-driven construction does not perform this kind of
algebraic simplification; it produces a faithful, mechanical
translation that works for *any* truth table without any analysis of
the function's structure. That trade-off — generality over efficiency —
is appropriate here because the brief is to handle arbitrary
four-bit functions, not to optimise specific ones [2].

#### Trade-offs and alternatives

The truth-table-driven construction has one obvious cost: it produces a
gate count proportional to the number of 1s in the truth table. For
balanced four-input functions this is 8 multi-controlled X gates, which
on a real device would decompose into a substantial number of CNOTs and
single-qubit gates. For a simulator this is negligible.

Two refinements are worth knowing about even though we do not use them
here:

- **Bernstein–Vazirani-style oracles.** When $f$ is a *linear* balanced
  function — one of the form $f(x) = a \cdot x \bmod 2$ for some
  bitmask $a$ — the oracle reduces to a few CNOTs from the input
  qubits whose bits in $a$ are 1, into the output qubit. This is much
  more efficient but only applies to linear $f$.
- **Phase-oracle formulation.** Some treatments skip the output qubit
  entirely and apply a phase oracle $|x\rangle \mapsto (-1)^{f(x)}
  |x\rangle$ directly to the input register. The two are equivalent
  given an output qubit prepared in $|-\rangle$, and the truth-table
  construction can be transposed to that style with minor changes.

Our construction handles both constant and balanced functions
uniformly without needing to know in advance which class $f$ belongs
to — exactly the property required for the algorithm to be honest:
the circuit is built from `f` alone, with no peeking at the answer.

### References

[1] A. Barenco et al., "Elementary gates for quantum computation,"
*Physical Review A*, vol. 52, no. 5, pp. 3457–3467, 1995.
https://doi.org/10.1103/PhysRevA.52.3457

[2] M. A. Nielsen and I. L. Chuang, *Quantum Computation and Quantum
Information*, 10th anniversary ed. Cambridge: Cambridge University Press,
2010, sec. 4.3 ("Controlled operations").

### Implementing `dj_oracle_from_function`

The function below takes either a Python callable of the form returned
by `random_constant_balanced` (a `BoolFunc4`) or a 16-bit truth table
integer, and returns the corresponding XOR oracle as a five-qubit
Qiskit circuit. Internally it follows the three-step encoding from the
previous cell exactly: extract the truth table, walk the "1" entries,
emit one $X$/$\text{MCX}$/$X$ sandwich per entry.

For convenience we accept either input form. Truth tables are slightly
faster to work with directly (no Python calls), but the algorithm is
identical either way.

In [149]:
def _function_to_truth_table(f: BoolFunc4) -> int:
    """Evaluate a BoolFunc4 on all 16 inputs and pack the result into a 16-bit integer.

    Bit ``i`` of the returned integer holds ``f`` applied to the four-bit
    input whose binary encoding equals ``i``, with ``x3`` as the most
    significant input bit (matching the convention from Problem 1).
    """
    table = 0
    for i in range(16):
        x3 = bool((i >> 3) & 1)
        x2 = bool((i >> 2) & 1)
        x1 = bool((i >> 1) & 1)
        x0 = bool(i & 1)
        if f(x3, x2, x1, x0):
            table |= (1 << i)
    return table


def dj_oracle_from_function(
    f: BoolFunc4 | int,
    n_inputs: int = 4,
) -> QuantumCircuit:
    """Build a Deutsch–Jozsa XOR oracle for a four-input Boolean function.

    The returned circuit has ``n_inputs + 1`` qubits: input qubits
    ``0..n_inputs-1`` hold the input bits ``(x0, x1, ..., x_{n-1})`` and
    qubit ``n_inputs`` holds the output. The unitary realised is

        |x>|y>  ->  |x>|y XOR f(x)>

    for every basis state ``|x>`` of the input register.

    Construction follows the three-step encoding documented in the
    previous cell:

    1. Extract a 16-bit truth table from ``f`` (or accept one directly).
    2. For each input ``x`` with ``f(x) = 1``, append the gate sandwich
       (X on zero-bit input qubits) | MCX(all inputs -> output) | (X on
       zero-bit input qubits) so that the output qubit is flipped
       exactly when the input register is ``|x>``.
    3. Bundle everything into one ``QuantumCircuit`` with a sensible name.

    Parameters
    ----------
    f : BoolFunc4 or int
        Either a four-argument Boolean callable (as produced by
        ``random_constant_balanced``) or a non-negative integer whose
        binary representation is the 16-bit truth table.
    n_inputs : int, default 4
        Number of input qubits. Defaults to 4 to match Problem 1; the
        construction generalises to any ``n``.

    Returns
    -------
    QuantumCircuit
        An ``(n_inputs + 1)``-qubit circuit implementing the XOR oracle.
    """
    # Step 1: normalise input to a truth-table integer.
    if isinstance(f, int):
        if not 0 <= f < (1 << (1 << n_inputs)):
            raise ValueError(
                f"truth-table integer out of range for n_inputs={n_inputs}"
            )
        table = f
    else:
        if n_inputs != 4:
            raise ValueError(
                "passing a callable is only supported for n_inputs=4 "
                "(matches the BoolFunc4 signature from Problem 1)."
            )
        table = _function_to_truth_table(f)

    output_qubit = n_inputs
    qc = QuantumCircuit(n_inputs + 1, name=f"U_f[0x{table:0{(2 ** n_inputs) // 4}X}]")

    input_qubits = list(range(n_inputs))

    # Step 2: emit one X | MCX | X sandwich per "1" entry in the truth
    # table.
    for x in range(2 ** n_inputs):
        if not ((table >> x) & 1):
            continue  # f(x) = 0: nothing to do.

        # Find which input qubits would be in |0> for this value of x.
        # We flip those before the MCX so all four controls trigger.
        zero_bit_qubits = [q for q in input_qubits if not ((x >> q) & 1)]

        for q in zero_bit_qubits:
            qc.x(q)
        qc.mcx(input_qubits, output_qubit)
        for q in zero_bit_qubits:
            qc.x(q)

    return qc

### Implementing `dj_circuit` and verifying on the four required functions

The Deutsch–Jozsa circuit follows the same five-step recipe as Deutsch's
algorithm in Problem 4, generalised to $n$ input qubits as derived in
Step 2 of this problem. The implementation below takes any oracle built
by `dj_oracle_from_function` and returns a complete, runnable circuit.

For the demonstration we run the circuit on:

- **Both constant functions**, $f(x) = 0$ (truth table $0x0000$) and
  $f(x) = 1$ (truth table $0xFFFF$).
- **Two balanced functions of our choosing**:
  - $f(x) = x_0$ (truth table $0xAAAA$) — the same example we worked
    through in Step 3 and used as an edge case in Problems 1 and 2.
  - $f(x) = x_0 \oplus x_1$ (truth table $0x6666$) — a non-trivial
    balanced function whose oracle requires every one of its eight
    multi-controlled gates, with no algebraic simplification possible
    by inspection.

The expected outcome on a noiseless simulator is deterministic: the
constant functions produce the bitstring `'0000'` on every shot, and
the balanced functions produce some non-zero bitstring on every shot
(which non-zero bitstring depends on the function's structure).

In [151]:
def dj_circuit(oracle: QuantumCircuit, n_inputs: int = 4) -> QuantumCircuit:
    """Build a Deutsch–Jozsa circuit using the supplied oracle.

    The oracle must be an ``(n_inputs + 1)``-qubit circuit implementing
    the XOR convention :math:`U_f|x\\rangle|y\\rangle = |x\\rangle|y
    \\oplus f(x)\\rangle`, with input qubits 0..n_inputs-1 and output
    qubit at index ``n_inputs`` — exactly the format produced by
    ``dj_oracle_from_function``.

    The returned circuit performs:

    1. Prepare the output qubit in :math:`|-\\rangle` (X then H).
    2. Apply Hadamard to every input qubit.
    3. Apply the oracle.
    4. Apply a second Hadamard to every input qubit.
    5. Measure each input qubit into a classical bit.

    Parameters
    ----------
    oracle : QuantumCircuit
        The XOR oracle circuit.
    n_inputs : int, default 4
        Number of input qubits. Must match the oracle.

    Returns
    -------
    QuantumCircuit
        A complete Deutsch–Jozsa circuit with ``n_inputs + 1`` quantum
        qubits and ``n_inputs`` classical bits.
    """
    if oracle.num_qubits != n_inputs + 1:
        raise ValueError(
            f"oracle has {oracle.num_qubits} qubits but n_inputs={n_inputs} "
            f"requires {n_inputs + 1} (n inputs + 1 output)."
        )

    output_qubit = n_inputs
    qc = QuantumCircuit(n_inputs + 1, n_inputs, name=f"DJ[{oracle.name}]")

    # Step 1: prepare the output qubit in |->.
    qc.x(output_qubit)
    qc.h(output_qubit)

    # Step 2: Hadamard on every input qubit, putting the input register
    # into the uniform superposition over all 2^n basis states.
    for q in range(n_inputs):
        qc.h(q)

    qc.barrier(label="oracle")

    # Step 3: apply the oracle. Composed in directly (rather than wrapped
    # in a gate box) for the same reason as Problem 4: the gates inside
    # the oracle remain visible when the circuit is drawn.
    qc.compose(oracle, qubits=list(range(n_inputs + 1)), inplace=True)

    qc.barrier(label="measure")

    # Step 4: second Hadamard on every input qubit. This is the
    # interference layer that converts the kicked-back phases into a
    # measurable bitstring.
    for q in range(n_inputs):
        qc.h(q)

    # Step 5: measure each input qubit into the corresponding classical
    # bit. The output qubit is left unmeasured: it has factored out as
    # |-> and is no longer entangled with the input register.
    for q in range(n_inputs):
        qc.measure(q, q)

    return qc

In [152]:
# The four required functions, expressed as truth-table integers so the
# oracle builder can consume them directly.
DJ_DEMO_CASES = [
    ("constant 0",       0x0000, "constant"),
    ("constant 1",       0xFFFF, "constant"),
    ("balanced f=x0",    0xAAAA, "balanced"),
    ("balanced x0 XOR x1", 0x6666, "balanced"),
]

dj_results: dict[str, dict] = {}

print(f"{'function':>22}  {'class':>9}  {'measured':>10}  {'decided':>9}  {'ok':>3}")
print("-" * 65)

n_shots = 1024
all_correct = True

for label, table, expected_class in DJ_DEMO_CASES:
    oracle = dj_oracle_from_function(table, n_inputs=4)
    circuit = dj_circuit(oracle, n_inputs=4)

    job = simulator.run(circuit, shots=n_shots)
    counts = job.result().get_counts()

    measured_bitstring = max(counts, key=counts.get)

    # Decision rule from Step 2's derivation:
    #   measured == '0000'  ->  f is constant
    #   measured != '0000'  ->  f is balanced
    decided_class = "constant" if measured_bitstring == "0000" else "balanced"

    correct = decided_class == expected_class
    all_correct = all_correct and correct

    dj_results[label] = {
        "table": table,
        "circuit": circuit,
        "counts": counts,
        "measured_bitstring": measured_bitstring,
        "decided_class": decided_class,
        "expected_class": expected_class,
    }

    print(
        f"{label:>22}  {expected_class:>9}  "
        f"{measured_bitstring!r:>10}  {decided_class:>9}  "
        f"{'✓' if correct else '✗':>3}"
    )

assert all_correct, "Deutsch–Jozsa gave the wrong answer for some function."
print()
print(f"All four demo functions classified correctly with a single oracle query each.")

# Sanity: every histogram must be concentrated on a single bitstring on
# a noiseless simulator. The exact bitstring observed for the balanced
# functions encodes information about their structure (which we discuss
# in the visualisation cell), but in every case it must be a single
# deterministic outcome.
for label, data in dj_results.items():
    assert len(data["counts"]) == 1, (
        f"{label}: expected a single-outcome histogram on noiseless "
        f"simulation, got {data['counts']}."
    )
print("All histograms are concentrated on a single bitstring (no shot noise).")

              function      class    measured    decided   ok
-----------------------------------------------------------------
            constant 0   constant      '0000'   constant    ✓
            constant 1   constant      '0000'   constant    ✓
         balanced f=x0   balanced      '0001'   balanced    ✓
    balanced x0 XOR x1   balanced      '0011'   balanced    ✓

All four demo functions classified correctly with a single oracle query each.
All histograms are concentrated on a single bitstring (no shot noise).


### Visualising the circuits and measurement results

We produce two kinds of visual artefact for each of the four demo
functions:

1. **Circuit drawings.** Showing the full Deutsch–Jozsa circuit with the
   oracle composed in directly makes the constant-vs-balanced structural
   difference visible at the gate level: the constant-zero circuit has
   no oracle gates at all, the constant-one circuit has a single $X$
   buried inside the oracle region, and the balanced circuits have eight
   multi-controlled gates each.

2. **Result histograms.** Each histogram is a single bar — the algorithm
   is exact, so all 1024 shots land in the same bin. The bin label tells
   us not just constant vs balanced but, for the balanced cases, also
   reveals structural information about the function (more on this in
   the next sub-section).

Both sets of figures are saved into the `img/` folder. Filenames are
keyed by truth-table integer so they sort sensibly when listed.

In [157]:
# Draw each Deutsch–Jozsa circuit and save it.
for label, data in dj_results.items():
    circuit = data["circuit"]
    table = data["table"]

    # fold=-1 keeps the (potentially wide) circuit on a single line in
    # the saved PNG; the inline display in the notebook will fold
    # automatically if needed.
    fig = circuit.draw(output="mpl", fold=-1)
    fig.suptitle(f"Deutsch–Jozsa: {label} (truth table 0x{table:04X})", y=1.02)

    save_path = f"img/dj_circuit_0x{table:04X}.png"
    fig.savefig(save_path, bbox_inches="tight", dpi=150)

    plt.show()
    print(f"  saved to {save_path}")

  saved to img/dj_circuit_0x0000.png
  saved to img/dj_circuit_0xFFFF.png
  saved to img/dj_circuit_0xAAAA.png
  saved to img/dj_circuit_0x6666.png


In [159]:
from qiskit.visualization import plot_histogram

# Plot a histogram for each oracle's result and save it.
for label, data in dj_results.items():
    counts = data["counts"]
    table = data["table"]
    expected = data["expected_class"]

    fig = plot_histogram(
        counts,
        title=f"Deutsch–Jozsa: {label} ({expected})",
        bar_labels=True,
    )

    save_path = f"img/dj_histogram_0x{table:04X}.png"
    fig.savefig(save_path, bbox_inches="tight", dpi=150)

    plt.show()
    print(f"  saved to {save_path}")

  saved to img/dj_histogram_0x0000.png
  saved to img/dj_histogram_0xFFFF.png
  saved to img/dj_histogram_0xAAAA.png
  saved to img/dj_histogram_0x6666.png


### What the histograms reveal

Two facts are immediate from the histograms:

- **Both constant functions produce the bitstring `'0000'`**, with all
  1024 shots concentrated in that single bin. This matches the boxed
  decision rule from Step 2: the all-zeros outcome is the certificate
  of constancy.

- **Both balanced functions produce a non-zero bitstring**, again with
  all 1024 shots concentrated in a single bin. For our two examples:

  - $f(x) = x_0$ measures `'0001'` — the bitstring with only $q_0$ set.
  - $f(x) = x_0 \oplus x_1$ measures `'0011'` — the bitstring with
    $q_0$ and $q_1$ set.

  The pattern is not a coincidence. For balanced functions of the form
  $f(x) = a \cdot x \pmod 2$, the algorithm measures **exactly the
  bitmask $a$**. This is the well-known **Bernstein–Vazirani**
  refinement of Deutsch–Jozsa [1], in which a single query recovers not
  just the constant-vs-balanced classification but the full linear
  structure of the function. Our circuit handles both algorithms with
  no modification — the only difference is what one chooses to read
  from the result.

For non-linear balanced functions, the measurement still produces a
non-zero bitstring (so the constant-vs-balanced decision is correct),
but the bitstring no longer corresponds to a unique linear mask.

### References

[1] E. Bernstein and U. Vazirani, "Quantum complexity theory," *SIAM
Journal on Computing*, vol. 26, no. 5, pp. 1411–1473, 1997.
https://doi.org/10.1137/S0097539796300921

### Summary of Problem 5

We have:

- presented the Deutsch–Jozsa algorithm as the natural $n$-input
  generalisation of Deutsch's algorithm and traced the joint state
  through the same five steps used in Problem 4, generalised to
  produce the $2^n$-term superposition and the all-zeros decision rule;
- explained how a classical Boolean function is encoded as a quantum
  oracle via the three-step recipe of extracting a truth table,
  emitting an $X$ / multi-controlled $X$ / $X$ sandwich for every
  "1" entry, and concatenating the results — a generic construction
  that works for any truth table without algebraic analysis;
- implemented `dj_oracle_from_function` as a small factory that
  accepts either a Python `BoolFunc4` callable (as produced by
  `random_constant_balanced` from Problem 1) or a 16-bit truth-table
  integer, and returns a five-qubit Qiskit oracle;
- implemented `dj_circuit` as a parametric Deutsch–Jozsa circuit
  factory generalising the Problem 4 implementation to $n$ input
  qubits, with defensive validation on the oracle's qubit count;
- run the circuit on the four required demonstration functions
  (the two constant functions plus $f(x) = x_0$ and $f(x) = x_0
  \oplus x_1$) and confirmed that each one is classified correctly
  with all 1024 shots concentrated on a single deterministic
  bitstring;
- visualised the four circuits and the four result histograms and
  saved each figure into the `img/` folder for offline reference;
- observed that for the linear balanced functions our algorithm
  measures exactly the bitmask $a$ in the representation $f(x) = a
  \cdot x \pmod 2$, which is the Bernstein–Vazirani refinement of
  Deutsch–Jozsa.

### Closing remarks: the complete classical-vs-quantum arc

This notebook has worked through a single, focused arc spanning all
five problems:

- **Problem 1** characterised the promise set of four-input
  constant-or-balanced Boolean functions (12,872 in total: 2 constant
  and 12,870 balanced) and built `random_constant_balanced` to sample
  uniformly from it. This established the *space of inputs* the rest
  of the notebook would operate on.

- **Problem 2** asked how hard the constant-vs-balanced decision is
  *classically*, derived the worst-case query bound of $2^{n-1}+1 =
  9$ via an adversary argument, and implemented an early-terminating
  classifier that achieves this bound while often deciding much
  faster on balanced inputs. This established the *classical
  baseline* the quantum approach has to beat.

- **Problem 3** stepped down to the single-input case, enumerated the
  four possible Boolean functions $f_0, f_1, f_2, f_3$, and built
  XOR oracles for all four in Qiskit. This established the *quantum
  primitive* — what a "function-as-circuit" actually looks like.

- **Problem 4** put those oracles to work inside Deutsch's original
  1985 algorithm, tracing the state vector through phase kickback and
  interference and confirming on the simulator that a single oracle
  query suffices to decide constant vs balanced for $n = 1$. This
  established the *quantum advantage* in its simplest form.

- **Problem 5** generalised that result to $n = 4$ — exactly the
  setting of Problem 1 — by implementing the Deutsch–Jozsa
  algorithm and demonstrating it on functions drawn from the same
  promise set the classical analysis of Problem 2 quantified at 9
  queries. The single-query quantum classification, contrasted with
  that 9-query classical lower bound, is the *exponential separation*
  the brief asked us to explore.

The exponential separation between deterministic classical and exact
quantum query complexity demonstrated here was the first formal proof
that quantum computers could provably outperform classical ones on a
well-defined task. It is a *theoretical* separation — Deutsch–Jozsa
itself has no known practical application — but it remains the
canonical first step in understanding more recent algorithms like
Shor's [1] and Grover's [2] that *do* solve practically important
problems with quantum advantages, and the survey of query-complexity
separations by Buhrman and de Wolf [3] places it in the broader
landscape that this notebook only briefly touched.

### References

[1] P. W. Shor, "Polynomial-time algorithms for prime factorization
and discrete logarithms on a quantum computer," *SIAM Journal on
Computing*, vol. 26, no. 5, pp. 1484–1509, 1997.
https://doi.org/10.1137/S0097539795293172

[2] L. K. Grover, "A fast quantum mechanical algorithm for database
search," in *Proceedings of the 28th Annual ACM Symposium on Theory
of Computing*, 1996, pp. 212–219.
https://doi.org/10.1145/237814.237866

[3] H. Buhrman and R. de Wolf, "Complexity measures and decision tree
complexity: a survey," *Theoretical Computer Science*, vol. 288,
no. 1, pp. 21–43, 2002.
https://doi.org/10.1016/S0304-3975(01)00144-X